# Estimativa de SOH em veículos elétricos — pipeline interno congelado

Esta versão (`SOH_ARTIGOV2.ipynb`) preserva o pipeline reproduzível até as análises de incerteza, antes dos estudos externos com os Datasets 1 e 3. Sua única entrada de dados é `data/data_real_EV_charge.pkl`, formado pela união dos eventos empilhados dos subconjuntos de treino e teste do fold 0 original.

## Escopo metodológico

- Unidade experimental: evento real de recarga.
- Target: `soh_top10_ref`, definido por uma referência retrospectiva individual.
- Validação principal: cinco folds estratificados por evento, preservando a comparabilidade com o pipeline original.
- Balanceamento: SMOTER condicionado, aplicado exclusivamente ao treino de cada fold quando previsto pelo cenário.
- Modelos comparados: 20 features legadas e seleção aninhada de 20 features.
- Análises de incerteza: dispersão das árvores (U1), conformal baseline (U2), conformal adaptativo (U3) e U3-Mondrian condicionado pelo gate.
- Sensibilidade: ensemble residual sem limpeza baseada no target.

O desenho avalia generalização para novos eventos na população observada; não representa validação estrita em veículos completamente desconhecidos. Nenhum resultado dos Datasets 1 ou 3 integra esta versão.


## 1. Ambiente e reprodutibilidade

As versões, seeds e parâmetros centrais são registrados antes do acesso aos dados. Nenhuma biblioteca é instalada ou atualizada pelo notebook.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import pickle
import platform
import random
import sys

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import openpyxl

try:
    import torch
except ImportError:
    torch = None

RANDOM_STATE = 42
N_SPLITS = 5
MIN_SNIPPETS = 10
TARGET_COL = "soh_top10_ref"
SOH_MIN = 0.920
SOH_MAX = 1.005
SOH_BIN_STEP = 0.005

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

python_executable = Path(sys.executable).resolve()
if "soh_gpu" not in str(python_executable.parent).lower():
    raise RuntimeError(f"Ambiente incorreto: {python_executable}. Use exclusivamente soh_gpu.")

environment_info = pd.DataFrame(
    {
        "componente": [
            "Python executable", "Python", "NumPy", "pandas", "scikit-learn",
            "SciPy", "joblib", "openpyxl", "PyTorch", "CUDA disponível", "CUDA version",
        ],
        "versão/valor": [
            str(python_executable), platform.python_version(), np.__version__, pd.__version__,
            sklearn.__version__, scipy.__version__, joblib.__version__, openpyxl.__version__,
            getattr(torch, "__version__", "não disponível"),
            bool(torch is not None and torch.cuda.is_available()),
            getattr(getattr(torch, "version", None), "cuda", None),
        ],
    }
)

display(environment_info)


## 2. Caminhos portáteis e identidade da entrada

Todos os caminhos são resolvidos a partir da pasta do notebook. O hash SHA-256 impede o uso silencioso de um arquivo diferente do documentado.


In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "SOH_ARTIGO":
    candidate = NOTEBOOK_DIR / "SOH_ARTIGO"
    if not candidate.is_dir():
        raise FileNotFoundError("Execute o notebook a partir da raiz do repositório ou de SOH_ARTIGO.")
    NOTEBOOK_DIR = candidate.resolve()

DATA_DIR = NOTEBOOK_DIR / "data"
CACHE_DIR = NOTEBOOK_DIR / "cache"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
AUDIT_DIR = OUTPUT_DIR / "audit"
for directory in (CACHE_DIR, OUTPUT_DIR, AUDIT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "data_real_EV_charge.pkl"
DATA_MANIFEST_PATH = DATA_DIR / "data_real_EV_charge.manifest.json"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Entrada ausente: {DATA_PATH}")
if not DATA_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Manifesto ausente: {DATA_MANIFEST_PATH}")

def sha256_file(path, chunk_size=2**20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(chunk_size):
            digest.update(block)
    return digest.hexdigest()

input_manifest = json.loads(DATA_MANIFEST_PATH.read_text(encoding="utf-8-sig"))
input_sha256 = sha256_file(DATA_PATH)
expected_sha256 = input_manifest["output"]["sha256"]
if input_sha256 != expected_sha256:
    raise RuntimeError("O hash do dataset não corresponde ao manifesto publicado.")

path_audit = pd.DataFrame(
    {
        "item": ["notebook_dir", "dataset", "manifesto", "SHA-256"],
        "valor": [str(NOTEBOOK_DIR), str(DATA_PATH), str(DATA_MANIFEST_PATH), input_sha256],
    }
)
display(path_audit)


## 3. Carregamento do conjunto de eventos

O payload contém uma lista de sequências e uma tabela de metadados. A posição `i` da lista deve representar exatamente o evento descrito pela linha `i` dos metadados.


In [ ]:
with DATA_PATH.open("rb") as handle:
    dataset = pickle.load(handle)

required_payload_keys = {"event_sequences", "event_meta"}
if not isinstance(dataset, dict) or not required_payload_keys.issubset(dataset):
    raise TypeError(f"Payload inválido. Chaves obrigatórias: {sorted(required_payload_keys)}")

event_sequences = dataset["event_sequences"]
event_meta = dataset["event_meta"].copy().reset_index(drop=True)

print(f"Sequências carregadas: {len(event_sequences):,}".replace(",", "."))
print(f"Linhas de metadados:   {len(event_meta):,}".replace(",", "."))


## 4. Auditoria estrutural e de alinhamento

As verificações abaixo são condições de parada. Uma divergência impede a continuação do pipeline.


In [ ]:
EVENT_KEY_COLS = ["car", "mileage", "charge_segment"]
EXPECTED_EVENTS = 12_128
EXPECTED_CARS = 83
EXPECTED_SEQUENCE_TAIL = (128, 8)

required_meta_columns = {
    *EVENT_KEY_COLS, "capacity", "n_snippets", "snippet_shape_0", "snippet_shape_1", "fold", "split"
}
missing_columns = sorted(required_meta_columns.difference(event_meta.columns))
if missing_columns:
    raise KeyError(f"Colunas obrigatórias ausentes: {missing_columns}")

sequence_shapes = [np.asarray(sequence).shape for sequence in event_sequences]
sequence_snippet_counts = np.asarray([shape[0] for shape in sequence_shapes], dtype=int)
metadata_snippet_counts = event_meta["n_snippets"].to_numpy(dtype=int)

checks = {
    "payload alinhado": len(event_sequences) == len(event_meta),
    "checkpoint de eventos": len(event_meta) == EXPECTED_EVENTS,
    "checkpoint de carros": event_meta["car"].nunique() == EXPECTED_CARS,
    "chaves únicas": not event_meta.duplicated(EVENT_KEY_COLS).any(),
    "n_snippets alinhado": np.array_equal(sequence_snippet_counts, metadata_snippet_counts),
    "mínimo de 10 snippets": bool(event_meta["n_snippets"].ge(MIN_SNIPPETS).all()),
    "shape interno (128, 8)": all(len(shape) == 3 and shape[1:] == EXPECTED_SEQUENCE_TAIL for shape in sequence_shapes),
    "capacidade finita": bool(np.isfinite(pd.to_numeric(event_meta["capacity"], errors="coerce")).all()),
    "capacidade positiva": bool(pd.to_numeric(event_meta["capacity"], errors="coerce").gt(0).all()),
}

failed_checks = [name for name, passed in checks.items() if not passed]
if failed_checks:
    raise AssertionError(f"Falhas na auditoria: {failed_checks}")

audit_summary = pd.DataFrame(
    [
        ("Eventos", EXPECTED_EVENTS, len(event_meta)),
        ("Carros", EXPECTED_CARS, event_meta["car"].nunique()),
        ("Chaves duplicadas", 0, int(event_meta.duplicated(EVENT_KEY_COLS).sum())),
        ("Sequências desalinhadas", 0, int(np.sum(sequence_snippet_counts != metadata_snippet_counts))),
        ("Eventos com menos de 10 snippets", 0, int(event_meta["n_snippets"].lt(MIN_SNIPPETS).sum())),
        ("Sequências com shape interno inválido", 0, int(sum(not (len(s) == 3 and s[1:] == EXPECTED_SEQUENCE_TAIL) for s in sequence_shapes))),
    ],
    columns=["checkpoint", "esperado", "obtido"],
)
audit_summary["status"] = np.where(audit_summary["esperado"].eq(audit_summary["obtido"]), "OK", "FALHA")

split_summary = (
    event_meta.groupby(["fold", "split"], dropna=False)
    .agg(eventos=("car", "size"), carros=("car", "nunique"), min_snippets=("n_snippets", "min"), max_snippets=("n_snippets", "max"))
    .reset_index()
)

snippet_summary = event_meta["n_snippets"].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]).rename("n_snippets").to_frame()

display(audit_summary)
display(split_summary)
display(snippet_summary)


In [ ]:
stage1_audit = {
    "stage": "01_input_audit",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "completed": True,
    "dataset_sha256": input_sha256,
    "events": int(len(event_meta)),
    "cars": int(event_meta["car"].nunique()),
    "metadata_shape": list(event_meta.shape),
    "duplicates": int(event_meta.duplicated(EVENT_KEY_COLS).sum()),
    "minimum_snippets_observed": int(event_meta["n_snippets"].min()),
    "checks": {name: bool(value) for name, value in checks.items()},
}

(AUDIT_DIR / "stage1_input_audit.json").write_text(
    json.dumps(stage1_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Etapa 1 concluída: entrada íntegra, alinhada e pronta para a extração de features.")


### Resultado da etapa

O conjunto de entrada foi aceito somente após verificação de identidade, cardinalidade, unicidade das chaves, shapes e alinhamento entre sequências e metadados. A próxima etapa implementará a extração reproduzível das features globais e por faixas de SOC.


## 5. Identidade determinística e ordenação canônica

Cada evento recebe um identificador calculado exclusivamente a partir de `(car, mileage, charge_segment)`. As sequências e os metadados são reordenados juntos; isso preserva o alinhamento e torna a execução independente da ordem física do arquivo.


In [ ]:
def canonical_number(value):
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return format(numeric, ".10g")

def event_key_text(row):
    return "|".join([
        canonical_number(row["car"]),
        canonical_number(row["mileage"]),
        canonical_number(row["charge_segment"]),
    ])

canonical_meta = event_meta.copy()
canonical_meta["__car_sort"] = pd.to_numeric(canonical_meta["car"], errors="raise")
canonical_meta["__mileage_sort"] = pd.to_numeric(canonical_meta["mileage"], errors="raise")
canonical_meta["__segment_sort"] = pd.to_numeric(canonical_meta["charge_segment"], errors="raise")
canonical_order = canonical_meta.sort_values(
    ["__car_sort", "__mileage_sort", "__segment_sort"], kind="stable"
).index.to_numpy()

event_sequences = [event_sequences[index] for index in canonical_order]
event_meta = event_meta.iloc[canonical_order].reset_index(drop=True)
event_meta.insert(
    0,
    "event_id",
    ["ev_" + hashlib.sha256(event_key_text(row).encode("utf-8")).hexdigest()[:20]
     for row in event_meta.to_dict("records")],
)

if not event_meta["event_id"].is_unique:
    raise AssertionError("Colisão de event_id")
if event_meta.duplicated(EVENT_KEY_COLS).any():
    raise AssertionError("Chaves duplicadas após ordenação")
if not np.array_equal(
    np.asarray([np.asarray(sequence).shape[0] for sequence in event_sequences]),
    event_meta["n_snippets"].to_numpy(dtype=int),
):
    raise AssertionError("Alinhamento perdido durante a ordenação")

event_identity = event_meta[["event_id", *EVENT_KEY_COLS, "fold", "split"]].copy()
event_identity.to_csv(AUDIT_DIR / "event_identity.csv", index=False)
display(event_identity.head())
print("IDs únicos:", event_identity["event_id"].nunique())


## 6. Features globais e por faixas de SOC

As definições abaixo preservam integralmente o conjunto legado: 148 descritores globais e 105 descritores condicionados às faixas de SOC 0–20%, 20–40%, 40–60%, 60–80% e 80–100%.


In [ ]:
from tqdm.auto import tqdm

SIGNAL_NAMES = ["volt", "current", "soc", "max_single_volt", "min_single_volt", "max_temp", "min_temp", "time"]
SOC_BINS = [(0,20),(20,40),(40,60),(60,80),(80,100)]
SOC_BIN_NAMES = ["SOC_0_20","SOC_20_40","SOC_40_60","SOC_60_80","SOC_80_100"]
GLOBAL_SIGNALS = ["volt","current","current_abs","soc","max_single_volt","min_single_volt","cell_imbalance","max_temp","min_temp","temp_mean","thermal_gradient","power_proxy"]
SOC_BIN_SIGNALS = ["current_abs","cell_imbalance","temp_mean","power_proxy","volt"]

def event_to_frame(event):
    event=np.asarray(event)
    if event.ndim!=3 or event.shape[1:]!=(128,8):
        raise ValueError(f"Shape de evento inválido: {event.shape}")
    frame=pd.DataFrame(event.reshape(-1,8),columns=SIGNAL_NAMES)
    frame["snippet_idx"]=np.repeat(np.arange(event.shape[0]),128)
    frame["step_idx"]=np.tile(np.arange(128),event.shape[0])
    return frame

def add_physical_signals(frame):
    frame=frame.copy()
    frame["current_abs"]=frame["current"].abs()
    frame["cell_imbalance"]=frame["max_single_volt"]-frame["min_single_volt"]
    frame["temp_mean"]=(frame["max_temp"]+frame["min_temp"])/2
    frame["thermal_gradient"]=frame["max_temp"]-frame["min_temp"]
    frame["power_proxy"]=frame["volt"]*frame["current_abs"]
    return frame

def safe_slope(values):
    values=np.asarray(values,dtype=float)
    return 0.0 if len(values)<2 else float(np.polyfit(np.arange(len(values)),values,1)[0])

def basic_statistics(values,prefix):
    values=np.asarray(values,dtype=float)
    return {
        f"{prefix}_mean":np.mean(values),f"{prefix}_std":np.std(values),
        f"{prefix}_min":np.min(values),f"{prefix}_max":np.max(values),
        f"{prefix}_range":np.max(values)-np.min(values),f"{prefix}_median":np.median(values),
        f"{prefix}_p05":np.percentile(values,5),f"{prefix}_p25":np.percentile(values,25),
        f"{prefix}_p75":np.percentile(values,75),f"{prefix}_p95":np.percentile(values,95),
        f"{prefix}_iqr":np.percentile(values,75)-np.percentile(values,25),
        f"{prefix}_slope":safe_slope(values),
    }

def extract_global_features(event):
    frame=add_physical_signals(event_to_frame(event))
    unique_time=np.sort(frame["time"].unique())
    dt=float(np.median(np.diff(unique_time))) if len(unique_time)>1 else 0.0
    result={"n_snippets":event.shape[0],"n_samples":len(frame),"dt_est":dt,"event_duration_proxy_s":len(frame)*dt}
    for signal in GLOBAL_SIGNALS:
        result.update(basic_statistics(frame[signal].to_numpy(),signal))
    return result

def soc_statistics(values,prefix):
    values=np.asarray(values,dtype=float)
    if len(values)==0:
        return {f"{prefix}_mean":np.nan,f"{prefix}_std":np.nan,f"{prefix}_p95":np.nan,f"{prefix}_max":np.nan}
    return {f"{prefix}_mean":np.mean(values),f"{prefix}_std":np.std(values),f"{prefix}_p95":np.percentile(values,95),f"{prefix}_max":np.max(values)}

def extract_soc_features(event):
    frame=add_physical_signals(event_to_frame(event));result={}
    for (lower,upper),name in zip(SOC_BINS,SOC_BIN_NAMES):
        part=frame.loc[frame["soc"].ge(lower)&frame["soc"].lt(upper)]
        result[f"{name}_n_points"]=len(part)
        for signal in SOC_BIN_SIGNALS:
            result.update(soc_statistics(part[signal].to_numpy(),f"{signal}_{name}"))
    return result

if len(extract_global_features(np.asarray(event_sequences[0])))!=148:
    raise AssertionError("Número inesperado de features globais")
if len(extract_soc_features(np.asarray(event_sequences[0])))!=105:
    raise AssertionError("Número inesperado de features por SOC")


## 7. Indicadores de resistência elétrica aparente

Os sinais foram obtidos durante recargas usuais, não em um ensaio controlado de pulso. Portanto, as novas variáveis são denominadas **indicadores de resistência aparente**, e não medições diretas da resistência interna eletroquímica.

- **Estática aparente:** coeficiente de corrente em `V ~ I + SOC + temperatura média`.
- **Estática por SOC:** o mesmo coeficiente dentro de cada faixa de SOC.
- **Dinâmica por degrau:** distribuição de `|ΔV/ΔI|` para `|ΔI| ≥ 0,5 A`, sem atravessar fronteiras de snippets.
- **Dinâmica ajustada:** coeficiente de `ΔI` em `ΔV ~ ΔI + ΔSOC + Δtemperatura`.

Os valores assinados, qualidade do ajuste e quantidade de observações são preservados para auditoria.


In [ ]:
DYNAMIC_CURRENT_STEP_A = 0.5
MIN_REGRESSION_POINTS = 30
MIN_DYNAMIC_REGRESSION_STEPS = 20

def linear_model_stats(y,predictors,min_points):
    y=np.asarray(y,dtype=float)
    x=np.column_stack([np.asarray(p,dtype=float) for p in predictors])
    mask=np.isfinite(y)&np.isfinite(x).all(axis=1)
    y=y[mask];x=x[mask]
    if len(y)<min_points or np.std(x[:,0])<1e-8:
        return np.nan,np.nan,np.nan,int(len(y))
    design=np.column_stack([np.ones(len(y)),x])
    coefficients=np.linalg.lstsq(design,y,rcond=None)[0]
    predicted=design@coefficients
    residual=y-predicted
    denominator=np.sum((y-y.mean())**2)
    r2=1-np.sum(residual**2)/denominator if denominator>0 else np.nan
    rmse=float(np.sqrt(np.mean(residual**2)))
    return float(coefficients[1]),float(r2),rmse,int(len(y))

def extract_resistance_features(event):
    event=np.asarray(event,dtype=float)
    frame=add_physical_signals(event_to_frame(event))
    signed,r2,rmse,n=linear_model_stats(
        frame["volt"],[frame["current"],frame["soc"],frame["temp_mean"]],MIN_REGRESSION_POINTS
    )
    result={
        "apparent_resistance_static_signed_ohm":signed,
        "apparent_resistance_static_abs_ohm":abs(signed) if np.isfinite(signed) else np.nan,
        "apparent_resistance_static_r2":r2,
        "apparent_resistance_static_rmse_v":rmse,
        "apparent_resistance_static_n_points":n,
    }
    for (lower,upper),name in zip(SOC_BINS,SOC_BIN_NAMES):
        part=frame.loc[frame["soc"].ge(lower)&frame["soc"].lt(upper)]
        coefficient,_,_,_=linear_model_stats(
            part["volt"],[part["current"],part["soc"],part["temp_mean"]],MIN_REGRESSION_POINTS
        )
        result[f"apparent_resistance_static_{name}_abs_ohm"]=abs(coefficient) if np.isfinite(coefficient) else np.nan

    voltage=event[:,:,0];current=event[:,:,1];soc=event[:,:,2]
    temperature=(event[:,:,5]+event[:,:,6])/2
    delta_v=np.diff(voltage,axis=1).reshape(-1)
    delta_i=np.diff(current,axis=1).reshape(-1)
    delta_soc=np.diff(soc,axis=1).reshape(-1)
    delta_temp=np.diff(temperature,axis=1).reshape(-1)
    valid=np.isfinite(delta_v)&np.isfinite(delta_i)&(np.abs(delta_i)>=DYNAMIC_CURRENT_STEP_A)
    ratios=np.abs(delta_v[valid]/delta_i[valid])
    total_steps=delta_i.size
    result.update({
        "apparent_resistance_dynamic_step_count":int(len(ratios)),
        "apparent_resistance_dynamic_step_fraction":float(len(ratios)/total_steps) if total_steps else 0.0,
        "apparent_resistance_dynamic_mean_ohm":float(np.mean(ratios)) if len(ratios) else np.nan,
        "apparent_resistance_dynamic_std_ohm":float(np.std(ratios)) if len(ratios) else np.nan,
        "apparent_resistance_dynamic_median_ohm":float(np.median(ratios)) if len(ratios) else np.nan,
        "apparent_resistance_dynamic_p25_ohm":float(np.percentile(ratios,25)) if len(ratios) else np.nan,
        "apparent_resistance_dynamic_p75_ohm":float(np.percentile(ratios,75)) if len(ratios) else np.nan,
        "apparent_resistance_dynamic_p95_ohm":float(np.percentile(ratios,95)) if len(ratios) else np.nan,
        "apparent_resistance_dynamic_max_ohm":float(np.max(ratios)) if len(ratios) else np.nan,
    })
    dyn_signed,dyn_r2,_,dyn_n=linear_model_stats(
        delta_v[valid],[delta_i[valid],delta_soc[valid],delta_temp[valid]],MIN_DYNAMIC_REGRESSION_STEPS
    )
    result.update({
        "apparent_resistance_dynamic_regression_signed_ohm":dyn_signed,
        "apparent_resistance_dynamic_regression_abs_ohm":abs(dyn_signed) if np.isfinite(dyn_signed) else np.nan,
        "apparent_resistance_dynamic_regression_r2":dyn_r2,
        "apparent_resistance_dynamic_regression_n_steps":dyn_n,
    })
    return result

RESISTANCE_FEATURE_COUNT=23
if len(extract_resistance_features(np.asarray(event_sequences[0])))!=RESISTANCE_FEATURE_COUNT:
    raise AssertionError("Número inesperado de features de resistência")


## 8. Smoke test da extração

Três eventos são processados antes da execução integral. O teste verifica contagens, chaves, alinhamento e ausência de valores infinitos sem criar ou substituir o cache completo.


In [ ]:
smoke_rows=[]
for index in range(3):
    sequence=np.asarray(event_sequences[index])
    smoke_rows.append({
        "event_id":event_meta.loc[index,"event_id"],
        "global_features":len(extract_global_features(sequence)),
        "soc_features":len(extract_soc_features(sequence)),
        "resistance_features":len(extract_resistance_features(sequence)),
        "n_snippets":sequence.shape[0],
    })
smoke_result=pd.DataFrame(smoke_rows)
assert smoke_result["global_features"].eq(148).all()
assert smoke_result["soc_features"].eq(105).all()
assert smoke_result["resistance_features"].eq(23).all()
display(smoke_result)


## 9. Extração integral e caches verificáveis

Nesta etapa pré-target, o cache base contém 259 colunas: seis identificadores/metadados e 253 features legadas (148 globais + 105 por SOC). As cinco variáveis auxiliares derivadas do target serão adicionadas somente na etapa seguinte, quando o checkpoint passará a 264 colunas. A tabela estendida adiciona `event_id` e 23 indicadores de resistência, totalizando 283 colunas. Valores ausentes são preservados e deverão ser imputados exclusivamente dentro do treino de cada fold.


In [ ]:
FEATURE_STAGE_VERSION="features-v2.0.0-resistance"
LEGACY_CACHE=CACHE_DIR/"features_base_12128x259.pkl"
LEGACY_MANIFEST=CACHE_DIR/"features_base_12128x259.manifest.json"
EXTENDED_CACHE=CACHE_DIR/"features_extended_12128x283.pkl"
EXTENDED_MANIFEST=CACHE_DIR/"features_extended_12128x283.manifest.json"

META_COLS=["fold","split","car","mileage","charge_segment","capacity"]
FORBIDDEN={"fold","split","car","charge_segment","capacity","first_idx","last_idx","event_id_local","label","snippet_shape_0","snippet_shape_1"}

def file_manifest(stage,frame,path,parameters):
    return {
        "stage":stage,"stage_version":FEATURE_STAGE_VERSION,"created_at_utc":datetime.now(timezone.utc).isoformat(),
        "completed":True,"parameters":parameters,"input_sha256":input_sha256,
        "shape":list(frame.shape),"columns":list(frame.columns),"events":int(len(frame)),
        "cars":int(frame["car"].nunique()),"output_sha256":sha256_file(path),"output_bytes":path.stat().st_size,
    }

def atomic_frame_cache(frame,path,manifest_path,stage,parameters):
    temporary=path.with_suffix(path.suffix+".tmp")
    frame.to_pickle(temporary);temporary.replace(path)
    manifest=file_manifest(stage,frame,path,parameters)
    temporary_manifest=manifest_path.with_suffix(manifest_path.suffix+".tmp")
    temporary_manifest.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
    temporary_manifest.replace(manifest_path)

def load_compatible_cache(path,manifest_path,expected_shape):
    if not(path.is_file() and manifest_path.is_file()): return None
    manifest=json.loads(manifest_path.read_text(encoding="utf-8"))
    if not(manifest.get("completed") is True and manifest.get("stage_version")==FEATURE_STAGE_VERSION and manifest.get("input_sha256")==input_sha256):
        raise RuntimeError(f"Cache incompatível: {path.name}")
    frame=pd.read_pickle(path)
    if list(frame.shape)!=manifest["shape"] or list(frame.columns)!=manifest["columns"] or frame.shape!=expected_shape:
        raise RuntimeError(f"Cache estruturalmente inválido: {path.name}")
    if sha256_file(path)!=manifest["output_sha256"]:
        raise RuntimeError(f"Hash inválido: {path.name}")
    return frame

features_legacy=load_compatible_cache(LEGACY_CACHE,LEGACY_MANIFEST,(EXPECTED_EVENTS,259))
features_extended=load_compatible_cache(EXTENDED_CACHE,EXTENDED_MANIFEST,(EXPECTED_EVENTS,283))

if features_legacy is None or features_extended is None:
    global_rows=[];soc_rows=[];resistance_rows=[]
    records=event_meta.to_dict("records")
    for sequence,row in tqdm(zip(event_sequences,records),total=len(records),desc="Extração de features",mininterval=2):
        sequence=np.asarray(sequence)
        global_rows.append({**row,**extract_global_features(sequence)})
        soc_rows.append({**row,**extract_soc_features(sequence)})
        resistance_rows.append({"event_id":row["event_id"],**{key:row[key] for key in EVENT_KEY_COLS},**extract_resistance_features(sequence)})

    global_frame=pd.DataFrame(global_rows)
    soc_frame=pd.DataFrame(soc_rows).fillna(0)
    resistance_features=pd.DataFrame(resistance_rows)
    meta_cols=[column for column in META_COLS if column in global_frame]
    global_cols=[column for column in global_frame if column not in FORBIDDEN and column not in ["event_id","mileage","charge_segment"] and pd.api.types.is_numeric_dtype(global_frame[column])]
    soc_cols=[column for column in soc_frame if column not in FORBIDDEN and column not in ["event_id","mileage","charge_segment"] and pd.api.types.is_numeric_dtype(soc_frame[column]) and column not in global_cols]
    features_legacy=pd.concat([
        global_frame[meta_cols].reset_index(drop=True),
        global_frame[global_cols].reset_index(drop=True),
        soc_frame[soc_cols].reset_index(drop=True),
    ],axis=1)
    if features_legacy.shape!=(EXPECTED_EVENTS,259):
        raise AssertionError(f"Checkpoint legado divergente: {features_legacy.shape}")
    if features_legacy.duplicated(EVENT_KEY_COLS).any():
        raise AssertionError("Duplicações nas features legadas")

    features_extended=(
        event_identity[["event_id",*EVENT_KEY_COLS]]
        .merge(features_legacy,on=EVENT_KEY_COLS,how="left",validate="one_to_one",sort=False)
        .merge(resistance_features,on=["event_id",*EVENT_KEY_COLS],how="left",validate="one_to_one",sort=False)
    )
    if features_extended.shape!=(EXPECTED_EVENTS,283):
        raise AssertionError(f"Checkpoint estendido divergente: {features_extended.shape}")
    if features_extended["event_id"].duplicated().any():
        raise AssertionError("Duplicações nas features estendidas")
    numeric=features_extended.select_dtypes(include=[np.number])
    if np.isinf(numeric.to_numpy()).any():
        raise AssertionError("Valores infinitos nas features")

    atomic_frame_cache(features_legacy,LEGACY_CACHE,LEGACY_MANIFEST,"pre_target_features",{"metadata_columns":6,"global":148,"soc":105,"pre_target_columns":259})
    atomic_frame_cache(features_extended,EXTENDED_CACHE,EXTENDED_MANIFEST,"extended_features",{"pre_target_columns":259,"event_id":1,"resistance_features":23,"dynamic_current_step_a":DYNAMIC_CURRENT_STEP_A})
    cache_status="construído"
else:
    cache_status="reutilizado"

feature_audit=pd.DataFrame([
    ["Eventos",EXPECTED_EVENTS,len(features_extended)],
    ["Features globais por evento",148,len(extract_global_features(np.asarray(event_sequences[0])))],
    ["Features por SOC por evento",105,len(extract_soc_features(np.asarray(event_sequences[0])))],
    ["Colunas base pré-target",259,features_legacy.shape[1]],
    ["Features de resistência",23,len([c for c in features_extended if c.startswith("apparent_resistance_")])],
    ["Colunas estendidas",283,features_extended.shape[1]],
    ["IDs duplicados",0,int(features_extended["event_id"].duplicated().sum())],
],columns=["checkpoint","esperado","obtido"])
feature_audit["status"]=np.where(feature_audit["esperado"].eq(feature_audit["obtido"]),"OK","FALHA")
resistance_columns=[column for column in features_extended if column.startswith("apparent_resistance_")]
resistance_missing=(features_extended[resistance_columns].isna().mean().mul(100).rename("ausentes_%").to_frame())

display(feature_audit)
display(resistance_missing)
print("Cache:",cache_status)


### Resultado da etapa

O pipeline agora dispõe de dois artefatos independentes: o conjunto base pré-target de 259 colunas e o conjunto estendido de 283 colunas, que preserva a identidade determinística e acrescenta 23 indicadores de resistência aparente. A próxima etapa construirá a referência Top-10 e os targets supervisionados, adicionando as cinco colunas auxiliares necessárias ao checkpoint legado de 264 colunas.


## 10. Referências de capacidade e definição dos targets

Esta seção reproduz duas referências históricas distintas, sem confundi-las:

1. `capacity_ref_car`: média das **10 primeiras** capacidades de cada carro na ordem canônica `(mileage, charge_segment)`. Essa referência alimenta cinco variáveis auxiliares legadas.
2. `capacity_ref_top10_car`: média das **10 maiores** capacidades observadas de cada carro em todo o conjunto. O target principal é `soh_top10_ref = capacity / capacity_ref_top10_car`.

A referência Top-10 é retrospectiva: usa observações futuras do mesmo carro. Ela é mantida para reproduzir o pipeline original e porque a capacidade nominal inicial não está disponível. Essa escolha deverá ser declarada como limitação metodológica no artigo.


In [ ]:
TARGET_STAGE_VERSION = "targets-v1.0.0-legacy-top10"
BASE_FEATURE_CACHE = CACHE_DIR / "features_base_12128x259.pkl"
EXTENDED_FEATURE_CACHE = CACHE_DIR / "features_extended_12128x283.pkl"
LEGACY_TARGET_CACHE = CACHE_DIR / "features_targets_legacy_12128x272.pkl"
LEGACY_TARGET_MANIFEST = CACHE_DIR / "features_targets_legacy_12128x272.manifest.json"
EXTENDED_TARGET_CACHE = CACHE_DIR / "features_targets_extended_12128x296.pkl"
EXTENDED_TARGET_MANIFEST = CACHE_DIR / "features_targets_extended_12128x296.manifest.json"

LEGACY_CONTEXT_COLUMNS = [
    "capacity_ref_car", "soh_relative", "capacity_smooth_causal_5",
    "soh_smooth_causal_5", "degradation_smooth_causal_5",
]
TOP10_COLUMNS = [
    "capacity_ref_top10_car", TARGET_COL, "soh_top10_ref_clipped",
    "soh_top10_causal_5", "soh_top10_causal_5_clipped",
]
LABEL_COLUMNS = [
    "label_top10_below_098", "label_top10_below_095", "label_top10_below_093",
]

def build_legacy_targets(frame):
    required = EVENT_KEY_COLS + ["capacity"]
    missing = [column for column in required if column not in frame]
    if missing:
        raise KeyError(f"Colunas ausentes para targets: {missing}")
    if frame.duplicated(EVENT_KEY_COLS).any():
        raise AssertionError("Chaves de evento duplicadas antes dos targets")

    data = frame[required].copy()
    data["__row_order"] = np.arange(len(data))
    data = data.sort_values(["car", "mileage", "charge_segment"], kind="stable").reset_index(drop=True)
    grouped = data.groupby("car", sort=False, group_keys=False)

    data["capacity_ref_car"] = grouped["capacity"].transform(lambda values: values.iloc[:10].mean())
    data["soh_relative"] = data["capacity"] / data["capacity_ref_car"]
    data["capacity_smooth_causal_5"] = grouped["capacity"].transform(
        lambda values: values.rolling(5, min_periods=1).median()
    )
    data["soh_smooth_causal_5"] = data["capacity_smooth_causal_5"] / data["capacity_ref_car"]
    data["degradation_smooth_causal_5"] = 1.0 - data["soh_smooth_causal_5"]

    data["capacity_ref_top10_car"] = grouped["capacity"].transform(
        lambda values: values.nlargest(10).mean()
    )
    data[TARGET_COL] = data["capacity"] / data["capacity_ref_top10_car"]
    data["soh_top10_ref_clipped"] = data[TARGET_COL].clip(0.0, 1.0)
    data["soh_top10_causal_5"] = data.groupby("car", sort=False)[TARGET_COL].transform(
        lambda values: values.rolling(5, min_periods=1).mean()
    )
    data["soh_top10_causal_5_clipped"] = data["soh_top10_causal_5"].clip(0.0, 1.0)
    data["label_top10_below_098"] = (data[TARGET_COL] < 0.98).astype("int8")
    data["label_top10_below_095"] = (data[TARGET_COL] < 0.95).astype("int8")
    data["label_top10_below_093"] = (data[TARGET_COL] < 0.93).astype("int8")

    if not np.isfinite(data[LEGACY_CONTEXT_COLUMNS + TOP10_COLUMNS].to_numpy()).all():
        raise AssertionError("Targets contêm NaN ou infinito")
    return data.sort_values("__row_order", kind="stable").drop(columns="__row_order").reset_index(drop=True)

def attach_targets(feature_frame, target_frame):
    additions = LEGACY_CONTEXT_COLUMNS + TOP10_COLUMNS + LABEL_COLUMNS
    return feature_frame.merge(
        target_frame[EVENT_KEY_COLS + additions],
        on=EVENT_KEY_COLS, how="left", validate="one_to_one", sort=False,
    )


### Checkpoints de dimensionalidade

As cinco variáveis auxiliares elevam o artefato pré-target de 259 para 264 colunas. Em seguida, cinco campos Top-10 e três labels históricos produzem o checkpoint final de 272 colunas. No ramo estendido, as 23 features de resistência e o `event_id` são preservados, resultando em 296 colunas.


In [ ]:
base_features = pd.read_pickle(BASE_FEATURE_CACHE)
extended_features = pd.read_pickle(EXTENDED_FEATURE_CACHE)
if base_features.shape != (EXPECTED_EVENTS, 259):
    raise AssertionError(f"Cache base incompatível: {base_features.shape}")
if extended_features.shape != (EXPECTED_EVENTS, 283):
    raise AssertionError(f"Cache estendido incompatível: {extended_features.shape}")

targets = build_legacy_targets(base_features)
legacy_264 = base_features.merge(
    targets[EVENT_KEY_COLS + LEGACY_CONTEXT_COLUMNS],
    on=EVENT_KEY_COLS, how="left", validate="one_to_one", sort=False,
)
legacy_272 = attach_targets(base_features, targets)
extended_296 = attach_targets(extended_features, targets)

expected_shapes = {
    "base_pre_target": (EXPECTED_EVENTS, 259),
    "legacy_with_context": (EXPECTED_EVENTS, 264),
    "legacy_final": (EXPECTED_EVENTS, 272),
    "extended_final": (EXPECTED_EVENTS, 296),
}
obtained_frames = {
    "base_pre_target": base_features,
    "legacy_with_context": legacy_264,
    "legacy_final": legacy_272,
    "extended_final": extended_296,
}
for name, expected_shape in expected_shapes.items():
    if obtained_frames[name].shape != expected_shape:
        raise AssertionError(f"Checkpoint {name}: {obtained_frames[name].shape} != {expected_shape}")
    if obtained_frames[name].duplicated(EVENT_KEY_COLS).any():
        raise AssertionError(f"Duplicações em {name}")

if not np.allclose(
    targets.groupby("car")["capacity_ref_top10_car"].nunique().to_numpy(), 1
):
    raise AssertionError("Referência Top-10 não é constante por carro")

target_input_hashes = {
    "features_base_sha256": sha256_file(BASE_FEATURE_CACHE),
    "features_extended_sha256": sha256_file(EXTENDED_FEATURE_CACHE),
}

def atomic_target_cache(frame, path, manifest_path, stage):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_pickle(temporary)
    temporary.replace(path)
    manifest = {
        "stage": stage,
        "stage_version": TARGET_STAGE_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "completed": True,
        "parameters": {
            "event_order": ["car", "mileage", "charge_segment"],
            "initial_reference": "mean_first_10_chronological_capacities",
            "target_reference": "mean_10_largest_capacities_retrospective",
            "causal_window": 5,
            "causal_capacity_statistic": "median",
            "causal_target_statistic": "mean",
            "label_thresholds": [0.98, 0.95, 0.93],
        },
        "input_sha256": target_input_hashes,
        "shape": list(frame.shape),
        "columns": list(frame.columns),
        "events": int(len(frame)),
        "cars": int(frame["car"].nunique()),
        "output_sha256": sha256_file(path),
        "output_bytes": path.stat().st_size,
    }
    temporary_manifest = manifest_path.with_suffix(manifest_path.suffix + ".tmp")
    temporary_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_manifest.replace(manifest_path)

atomic_target_cache(legacy_272, LEGACY_TARGET_CACHE, LEGACY_TARGET_MANIFEST, "legacy_targets")
atomic_target_cache(extended_296, EXTENDED_TARGET_CACHE, EXTENDED_TARGET_MANIFEST, "extended_targets_resistance")


### Auditoria dos targets

Além dos shapes, são apresentados o intervalo do target, as referências por carro e a distribuição dos três labels. A comparação com o Gold histórico, quando disponível, é somente uma auditoria: o arquivo histórico não é modificado nem usado para construir os novos targets.


In [ ]:
checkpoint_table = pd.DataFrame([
    [name, expected_shapes[name], obtained_frames[name].shape,
     "OK" if expected_shapes[name] == obtained_frames[name].shape else "FALHA"]
    for name in expected_shapes
], columns=["checkpoint", "esperado", "obtido", "status"])

target_summary = targets[["capacity_ref_car", "capacity_ref_top10_car", TARGET_COL]].describe().T
label_summary = pd.DataFrame({
    "positivos": legacy_272[LABEL_COLUMNS].sum().astype(int),
    "percentual_%": legacy_272[LABEL_COLUMNS].mean().mul(100),
})
reference_summary = targets.groupby("car", sort=True).agg(
    eventos=("capacity", "size"),
    capacidade_ref_primeiras10=("capacity_ref_car", "first"),
    capacidade_ref_maiores10=("capacity_ref_top10_car", "first"),
    soh_min=(TARGET_COL, "min"),
    soh_max=(TARGET_COL, "max"),
).reset_index()

display(checkpoint_table)
display(target_summary)
display(label_summary)
display(reference_summary.head(10))

legacy_gold_path = NOTEBOOK_DIR.parent / "PIPELINE COMPLETO" / "cache" / "gold_unified.pkl"
historical_comparison = None
if legacy_gold_path.is_file():
    historical_gold = pd.read_pickle(legacy_gold_path)
    comparison_columns = EVENT_KEY_COLS + [TARGET_COL]
    if set(comparison_columns).issubset(historical_gold.columns):
        reconstructed_for_audit = legacy_272[comparison_columns].copy()
        historical_for_audit = historical_gold[comparison_columns].copy()
        for key in EVENT_KEY_COLS:
            reconstructed_for_audit[key] = pd.to_numeric(reconstructed_for_audit[key], errors="raise")
            historical_for_audit[key] = pd.to_numeric(historical_for_audit[key], errors="raise")
        historical_comparison = reconstructed_for_audit.merge(
            historical_for_audit, on=EVENT_KEY_COLS,
            suffixes=("_reconstruido", "_historico"), validate="one_to_one",
        )
        delta = (
            historical_comparison[f"{TARGET_COL}_reconstruido"]
            - historical_comparison[f"{TARGET_COL}_historico"]
        ).abs()
        print(
            "Comparação Gold histórico:", len(delta), "eventos;",
            "diferença máxima =", f"{delta.max():.3e}",
            "; diferenças > 1e-12 =", int((delta > 1e-12).sum()),
        )

print("Cache legado:", LEGACY_TARGET_CACHE.name, legacy_272.shape)
print("Cache estendido:", EXTENDED_TARGET_CACHE.name, extended_296.shape)


### Resultado da etapa

Os targets e labels do pipeline original foram reconstruídos de forma determinística, mantendo a referência Top-10 retrospectiva e separando-a explicitamente da referência baseada nas dez primeiras observações. O artefato legado de 272 colunas permanece disponível para comparabilidade; o artefato estendido de 296 colunas acrescenta os indicadores de resistência aparente.

A próxima etapa reconstruirá o fold 3 original e a limpeza Gold 1. A análise de sensibilidade manterá, em paralelo, uma versão sem a limpeza baseada no target.


## 11. Limpeza Clean 1 em todos os eventos

A limpeza é aplicada diretamente aos 12.128 eventos, sem recuperar a partição histórica do fold 3. Os eventos são ordenados por `(car, mileage, charge_segment)` e processados por trajetória individual.

Esta variante é denominada **Clean 1 pooled**. Ela preserva as regras e os limiares relativos do método legado, mas estima os quantis e as estatísticas robustas usando o conjunto completo. Portanto, não é apresentada como reprodução exata da limpeza histórica treino/teste. Como os critérios usam o próprio SOH, o conjunto sem essa limpeza será mantido para a análise de sensibilidade.


In [ ]:
CLEAN_STAGE_VERSION = "clean-v1.0.0-pooled"
LEGACY_272_CACHE = CACHE_DIR / "features_targets_legacy_12128x272.pkl"
EXTENDED_296_CACHE = CACHE_DIR / "features_targets_extended_12128x296.pkl"
CLEAN_LEGACY_CACHE = CACHE_DIR / "clean1_pooled_legacy.pkl"
CLEAN_LEGACY_MANIFEST = CACHE_DIR / "clean1_pooled_legacy.manifest.json"
CLEAN_EXTENDED_CACHE = CACHE_DIR / "clean1_pooled_extended.pkl"
CLEAN_EXTENDED_MANIFEST = CACHE_DIR / "clean1_pooled_extended.manifest.json"
CLEAN_REMOVED_CACHE = CACHE_DIR / "clean1_removed_events.pkl"
CLEAN_REMOVED_MANIFEST = CACHE_DIR / "clean1_removed_events.manifest.json"

PHYSICAL_AUDIT_FEATURES = [
    "min_temp_p75", "max_temp_mean", "temp_mean_SOC_40_60_mean",
    "min_single_volt_slope", "temp_mean_SOC_60_80_mean", "max_temp_p05",
    "volt_slope", "max_temp_p25", "max_single_volt_slope", "min_temp_p95",
    "min_temp_mean", "soc_slope",
]
CLEAN_DROP_COLUMNS = [
    "capacity_ref_car", "soh_relative", "capacity_smooth_causal_5",
    "soh_smooth_causal_5", "degradation_smooth_causal_5",
    "label_top10_below_098", "label_top10_below_095", "label_top10_below_093",
]

def add_causal_smooths(frame, windows=(5, 10, 20), suffix=""):
    data = frame.sort_values(EVENT_KEY_COLS, kind="stable").reset_index(drop=True).copy()
    for window in windows:
        data[f"soh_top10_causal_{window}{suffix}"] = data.groupby("car", sort=False)[TARGET_COL].transform(
            lambda values: values.rolling(window, min_periods=1).mean()
        )
        data[f"capacity_top10_causal_{window}{suffix}"] = data.groupby("car", sort=False)["capacity"].transform(
            lambda values: values.rolling(window, min_periods=1).mean()
        )
    return data

def add_causal_deltas(frame, windows=(5, 10, 20), suffix=""):
    data = frame.copy()
    for window in windows:
        for base, value_column in (("soh", TARGET_COL), ("capacity", "capacity")):
            causal_column = f"{base}_top10_causal_{window}{suffix}"
            delta_column = f"delta_{base}_top10_causal_{window}{suffix}"
            data[delta_column] = data[value_column] - data[causal_column]
            data[f"abs_{delta_column}"] = data[delta_column].abs()
    return data

def add_neighbor_features(frame, suffix=""):
    data = frame.sort_values(EVENT_KEY_COLS, kind="stable").reset_index(drop=True).copy()
    grouped = data.groupby("car", sort=False)
    pairs = [("soh", TARGET_COL)] if suffix else [("soh", TARGET_COL), ("capacity", "capacity")]
    for base, value_column in pairs:
        previous = f"{base}_prev{suffix}"
        following = f"{base}_next{suffix}"
        mean_column = f"{base}_neighbor_mean{suffix}"
        delta_column = f"delta_{base}_neighbor{suffix}"
        data[previous] = grouped[value_column].shift(1)
        data[following] = grouped[value_column].shift(-1)
        data[mean_column] = data[[previous, following]].mean(axis=1)
        data[delta_column] = data[value_column] - data[mean_column]
        data[f"abs_{delta_column}"] = data[delta_column].abs()
        data[f"neighbor_{base}_gap{suffix}"] = (data[previous] - data[following]).abs()
    data[f"neighbor_consistency_score_raw{suffix}"] = (
        data[f"abs_delta_soh_neighbor{suffix}"] / (data[f"neighbor_soh_gap{suffix}"] + 1e-6)
    )
    if not suffix:
        data["has_both_neighbors"] = data["soh_prev"].notna() & data["soh_next"].notna()
        data.loc[~data["has_both_neighbors"], "neighbor_consistency_score_raw"] = np.nan
    return data

def add_target_score(frame, thresholds, weights, version):
    data = frame.copy()
    total_column = f"target_anomaly_score_{version}"
    data[total_column] = 0.0
    for column, threshold in thresholds.items():
        score_column = column.replace("abs_delta_", "score_")
        if not np.isfinite(threshold) or threshold <= 0:
            raise ValueError(f"Limiar inválido em {column}: {threshold}")
        data[score_column] = (data[column] / threshold).clip(0.0, 1.0)
        data[total_column] += weights[column] * data[score_column]
    return data

def reconstruct_clean1_pooled(frame):
    source = frame.drop(columns=CLEAN_DROP_COLUMNS, errors="ignore").copy()
    source = source.sort_values(EVENT_KEY_COLS, kind="stable").reset_index(drop=True)
    source_ids = set(source["event_id"])
    audit_rows = []

    car_counts = source.groupby("car", sort=False).size()
    valid_cars = car_counts[car_counts >= 20].index
    removed_small = source.loc[~source["car"].isin(valid_cars)].copy()
    for row in removed_small[["event_id", "car", "mileage", "charge_segment", TARGET_COL]].to_dict("records"):
        audit_rows.append({**row, "removal_stage": "minimum_20_events_per_car", "removal_score": np.nan, "removal_threshold": 20.0})
    clean = source[source["car"].isin(valid_cars)].copy()

    clean = add_neighbor_features(add_causal_deltas(add_causal_smooths(clean)))
    neighbor_p95 = clean["abs_delta_soh_neighbor"].quantile(0.95)
    neighbor_gap_p75 = clean["neighbor_soh_gap"].quantile(0.75)
    clean["is_neighbor_target_anomaly"] = (
        clean["has_both_neighbors"]
        & (clean["abs_delta_soh_neighbor"] >= neighbor_p95)
        & (clean["neighbor_soh_gap"] <= neighbor_gap_p75)
    )
    score_columns_v1 = [
        "abs_delta_soh_neighbor", "abs_delta_soh_top10_causal_5",
        "abs_delta_soh_top10_causal_10",
    ]
    thresholds_v1 = {column: clean[column].quantile(0.99) for column in score_columns_v1}
    weights_v1 = dict(zip(score_columns_v1, (0.50, 0.35, 0.15)))
    clean = add_target_score(clean, thresholds_v1, weights_v1, "v1")
    clean.loc[clean["is_neighbor_target_anomaly"], "target_anomaly_score_v1"] = (
        clean.loc[clean["is_neighbor_target_anomaly"], "target_anomaly_score_v1"].clip(lower=0.8)
    )
    clean["target_anomaly_level"] = "normal"
    clean.loc[clean["target_anomaly_score_v1"] >= 0.5, "target_anomaly_level"] = "attention"
    clean.loc[clean["target_anomaly_score_v1"] >= 0.8, "target_anomaly_level"] = "high_anomaly"
    removed_v1 = clean[clean["target_anomaly_level"].eq("high_anomaly")].copy()
    for row in removed_v1[["event_id", "car", "mileage", "charge_segment", TARGET_COL, "target_anomaly_score_v1"]].to_dict("records"):
        score = row.pop("target_anomaly_score_v1")
        audit_rows.append({**row, "removal_stage": "target_anomaly_v1", "removal_score": score, "removal_threshold": 0.8})
    clean = clean[~clean["target_anomaly_level"].eq("high_anomaly")].copy()

    clean = add_neighbor_features(
        add_causal_deltas(add_causal_smooths(clean, (5, 10), "_v2"), (5, 10), "_v2"), "_v2"
    )
    score_columns_v2 = [
        "abs_delta_soh_neighbor_v2", "abs_delta_soh_top10_causal_5_v2",
        "abs_delta_soh_top10_causal_10_v2",
    ]
    thresholds_v2 = {column: clean[column].quantile(0.99) for column in score_columns_v2}
    weights_v2 = dict(zip(score_columns_v2, (0.50, 0.35, 0.15)))
    clean = add_target_score(clean, thresholds_v2, weights_v2, "v2")
    clean["target_anomaly_level_v2"] = "normal"
    clean.loc[clean["target_anomaly_score_v2"] >= 0.7, "target_anomaly_level_v2"] = "attention"
    clean.loc[clean["target_anomaly_score_v2"] >= 0.91, "target_anomaly_level_v2"] = "high_anomaly"
    removed_v2 = clean[clean["target_anomaly_level_v2"].eq("high_anomaly")].copy()
    for row in removed_v2[["event_id", "car", "mileage", "charge_segment", TARGET_COL, "target_anomaly_score_v2"]].to_dict("records"):
        score = row.pop("target_anomaly_score_v2")
        audit_rows.append({**row, "removal_stage": "target_anomaly_v2", "removal_score": score, "removal_threshold": 0.91})
    clean = clean[~clean["target_anomaly_level_v2"].eq("high_anomaly")].copy()

    median = clean[PHYSICAL_AUDIT_FEATURES].median()
    mad = clean[PHYSICAL_AUDIT_FEATURES].sub(median).abs().median().replace(0, 1e-9)
    robust_z = clean[PHYSICAL_AUDIT_FEATURES].sub(median).div(mad).abs()
    robust_z.columns = [f"rz_v3_{column}" for column in PHYSICAL_AUDIT_FEATURES]
    clean[robust_z.columns] = robust_z
    clean["feature_anomaly_score_rz_mean_v3"] = robust_z.mean(axis=1)
    clean["feature_anomaly_score_rz_max_v3"] = robust_z.max(axis=1)
    clean["feature_anomaly_n_features_rz_gt_3_v3"] = (robust_z > 3).sum(axis=1)
    clean["feature_anomaly_n_features_rz_gt_5_v3"] = (robust_z > 5).sum(axis=1)
    clean["charge_profile_current_v1"] = pd.cut(
        clean["current_abs_p95"], [-np.inf, 6, 12, 22, np.inf],
        labels=["slow", "medium", "fast", "very_fast"], right=False,
    ).astype(str)
    clean["charge_profile_current_code_v1"] = clean["charge_profile_current_v1"].map(
        {"slow": 0, "medium": 1, "fast": 2, "very_fast": 3}
    )
    clean["cleaning_scope"] = "pooled_all_events"
    clean["sample_origin"] = "real"

    removed = pd.DataFrame(audit_rows)
    if not removed.empty:
        removed = removed.sort_values(["car", "mileage", "charge_segment"], kind="stable").reset_index(drop=True)
    if len(clean) + len(removed) != len(source):
        raise AssertionError("Balanço de eventos inconsistente na limpeza")
    if set(clean["event_id"]) & set(removed["event_id"]):
        raise AssertionError("Evento simultaneamente mantido e removido")
    if set(clean["event_id"]) | set(removed["event_id"]) != source_ids:
        raise AssertionError("Cobertura incompleta dos eventos")
    return clean.reset_index(drop=True), removed, {
        "neighbor_p95": float(neighbor_p95), "neighbor_gap_p75": float(neighbor_gap_p75),
        "thresholds_v1": {key: float(value) for key, value in thresholds_v1.items()},
        "thresholds_v2": {key: float(value) for key, value in thresholds_v2.items()},
    }


## 12. Execução, auditoria e ramo sem limpeza

O ramo `no_target_cleaning` preserva todos os eventos e será usado para quantificar a sensibilidade dos resultados à filtragem baseada no target. A versão estendida da limpeza é formada por junção um-para-um via `event_id`, preservando as 23 features de resistência aparente.


In [ ]:
legacy_272 = pd.read_pickle(LEGACY_272_CACHE)
extended_296 = pd.read_pickle(EXTENDED_296_CACHE)
if legacy_272.shape != (EXPECTED_EVENTS, 272) or extended_296.shape != (EXPECTED_EVENTS, 296):
    raise AssertionError("Caches de entrada incompatíveis com a limpeza")

legacy_with_id = legacy_272.merge(
    extended_296[EVENT_KEY_COLS + ["event_id"]],
    on=EVENT_KEY_COLS, how="left", validate="one_to_one", sort=False,
)
clean1_legacy, clean1_removed_events, clean_parameters = reconstruct_clean1_pooled(legacy_with_id)

resistance_columns = [column for column in extended_296 if column.startswith("apparent_resistance_")]
clean1_extended = clean1_legacy.merge(
    extended_296[["event_id"] + resistance_columns],
    on="event_id", how="left", validate="one_to_one", sort=False,
)
no_target_cleaning_legacy = legacy_272.copy()
no_target_cleaning_extended = extended_296.copy()

if clean1_legacy["event_id"].duplicated().any() or clean1_extended["event_id"].duplicated().any():
    raise AssertionError("IDs duplicados após a limpeza")
if len(clean1_extended) != len(clean1_legacy):
    raise AssertionError("Desalinhamento do ramo estendido")

clean_input_hashes = {
    "legacy_272_sha256": sha256_file(LEGACY_272_CACHE),
    "extended_296_sha256": sha256_file(EXTENDED_296_CACHE),
}

def atomic_clean_cache(frame, path, manifest_path, stage):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_pickle(temporary)
    temporary.replace(path)
    manifest = {
        "stage": stage, "stage_version": CLEAN_STAGE_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "completed": True,
        "parameters": {
            "scope": "all_events_pooled", "minimum_events_per_car": 20,
            "v1_score_threshold": 0.8, "v2_score_threshold": 0.91,
            "quantile_fit_scope": "all_events", **clean_parameters,
        },
        "input_sha256": clean_input_hashes, "shape": list(frame.shape),
        "columns": list(frame.columns), "events": int(len(frame)),
        "cars": int(frame["car"].nunique()) if "car" in frame else None,
        "output_sha256": sha256_file(path), "output_bytes": path.stat().st_size,
    }
    temporary_manifest = manifest_path.with_suffix(manifest_path.suffix + ".tmp")
    temporary_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_manifest.replace(manifest_path)

atomic_clean_cache(clean1_legacy, CLEAN_LEGACY_CACHE, CLEAN_LEGACY_MANIFEST, "clean1_pooled_legacy")
atomic_clean_cache(clean1_extended, CLEAN_EXTENDED_CACHE, CLEAN_EXTENDED_MANIFEST, "clean1_pooled_extended")
atomic_clean_cache(clean1_removed_events, CLEAN_REMOVED_CACHE, CLEAN_REMOVED_MANIFEST, "clean1_removed_events")


In [ ]:
clean_checkpoints = pd.DataFrame([
    ["eventos antes da limpeza", EXPECTED_EVENTS, len(legacy_272)],
    ["eventos mantidos", None, len(clean1_legacy)],
    ["eventos removidos", None, len(clean1_removed_events)],
    ["balanço de eventos", EXPECTED_EVENTS, len(clean1_legacy) + len(clean1_removed_events)],
    ["carros antes", 83, legacy_272["car"].nunique()],
    ["carros depois", None, clean1_legacy["car"].nunique()],
    ["features de resistência preservadas", 23, len(resistance_columns)],
], columns=["checkpoint", "esperado", "obtido"])
clean_checkpoints["status"] = np.where(
    clean_checkpoints["esperado"].isna() | clean_checkpoints["esperado"].eq(clean_checkpoints["obtido"]),
    "OK", "FALHA",
)

removals_by_stage = clean1_removed_events.groupby("removal_stage", dropna=False).size().rename("eventos").to_frame()
removals_by_car = clean1_removed_events.groupby("car").size().sort_values(ascending=False).rename("removidos").to_frame()
soh_bins = pd.IntervalIndex.from_breaks([-np.inf, 0.90, 0.93, 0.95, 0.98, 1.00, np.inf], closed="left")
removal_soh_distribution = pd.cut(clean1_removed_events[TARGET_COL], soh_bins).value_counts(sort=False).rename("removidos").to_frame()
before_after = pd.DataFrame({
    "antes": legacy_272[TARGET_COL].describe(),
    "depois": clean1_legacy[TARGET_COL].describe(),
})

display(clean_checkpoints)
display(removals_by_stage)
display(removals_by_car.head(15))
display(removal_soh_distribution)
display(before_after)
print("Clean legado:", clean1_legacy.shape)
print("Clean estendido:", clean1_extended.shape)
print("Controle sem limpeza legado:", no_target_cleaning_legacy.shape)
print("Controle sem limpeza estendido:", no_target_cleaning_extended.shape)


### Resultado da etapa

Foram produzidos dois ramos metodológicos:

- `clean1_pooled`: aplicação das regras legadas de limpeza ao conjunto completo;
- `no_target_cleaning`: todos os 12.128 eventos, sem filtragem derivada do SOH.

Na modelagem, os mesmos folds serão aplicados aos dois ramos. Assim, a diferença de desempenho poderá ser atribuída à limpeza, mantendo iguais o particionamento, a seleção de features, o pré-processamento e o estimador.


## 13. Faixa oficial de SOH e população de avaliação

Como no pipeline original, a modelagem considera `0,920 ≤ soh_top10_ref ≤ 1,005`. O filtro é aplicado separadamente da Clean 1 para que seja possível auditar quantos eventos cada regra remove.

O ramo sem limpeza define a população-mestre. O ramo Clean é um subconjunto dessa população e herdará exatamente a mesma atribuição de fold para cada `event_id` compartilhado.


In [ ]:
from sklearn.model_selection import StratifiedKFold

CV_STAGE_VERSION = "cv-v1.0.0-event-stratified-shared"
CV_RANDOM_STATE = 42
CV_N_SPLITS = 5
CV_SOH_MIN = 0.920
CV_SOH_MAX = 1.005
CV_SOH_STEP = 0.005
CV_ASSIGNMENTS_PATH = CACHE_DIR / "cv_event_folds_master.csv"
CV_MANIFEST_PATH = CACHE_DIR / "cv_event_folds_master.manifest.json"

no_clean_legacy = pd.read_pickle(CACHE_DIR / "features_targets_legacy_12128x272.pkl")
no_clean_extended = pd.read_pickle(CACHE_DIR / "features_targets_extended_12128x296.pkl")
clean_legacy = pd.read_pickle(CACHE_DIR / "clean1_pooled_legacy.pkl")
clean_extended = pd.read_pickle(CACHE_DIR / "clean1_pooled_extended.pkl")

no_clean_legacy = no_clean_legacy.merge(
    no_clean_extended[EVENT_KEY_COLS + ["event_id"]],
    on=EVENT_KEY_COLS, how="left", validate="one_to_one", sort=False,
)
for name, frame in {
    "no_target_cleaning_legacy": no_clean_legacy,
    "no_target_cleaning_extended": no_clean_extended,
    "clean1_pooled_legacy": clean_legacy,
    "clean1_pooled_extended": clean_extended,
}.items():
    if frame["event_id"].isna().any() or frame["event_id"].duplicated().any():
        raise AssertionError(f"Identidade inválida em {name}")

def official_soh_filter(frame):
    return frame.loc[
        pd.to_numeric(frame[TARGET_COL], errors="raise").between(CV_SOH_MIN, CV_SOH_MAX, inclusive="both")
    ].copy().reset_index(drop=True)

cv_no_target_cleaning_legacy = official_soh_filter(no_clean_legacy)
cv_no_target_cleaning_extended = official_soh_filter(no_clean_extended)
cv_clean1_pooled_legacy = official_soh_filter(clean_legacy)
cv_clean1_pooled_extended = official_soh_filter(clean_extended)

filter_audit = pd.DataFrame([
    ["no_target_cleaning", len(no_clean_legacy), len(cv_no_target_cleaning_legacy), len(no_clean_legacy)-len(cv_no_target_cleaning_legacy)],
    ["clean1_pooled", len(clean_legacy), len(cv_clean1_pooled_legacy), len(clean_legacy)-len(cv_clean1_pooled_legacy)],
], columns=["cenario", "antes_filtro_soh", "depois_filtro_soh", "removidos_filtro_soh"])
display(filter_audit)


## 14. Cinco folds estratificados por evento

Os folds medem generalização para novos eventos de carros já observados. Eventos do mesmo carro podem aparecer no treino e na validação; essa interpretação será declarada no artigo.

A atribuição é criada uma única vez sobre a população sem limpeza e depois propagada por `event_id`. Isso mantém constantes os folds na comparação de sensibilidade. Consequentemente, as atribuições não são numericamente idênticas às do pipeline histórico, que eram geradas depois da limpeza do fold 3.


In [ ]:
CV_SOH_EDGES = np.round(
    np.arange(CV_SOH_MIN, CV_SOH_MAX + CV_SOH_STEP, CV_SOH_STEP), 6
)
if CV_SOH_EDGES[-1] < CV_SOH_MAX:
    CV_SOH_EDGES = np.append(CV_SOH_EDGES, CV_SOH_MAX)

master_population = cv_no_target_cleaning_legacy[[
    "event_id", "car", "mileage", "charge_segment", TARGET_COL
]].copy()
master_population["soh_bin"] = pd.cut(
    master_population[TARGET_COL], bins=CV_SOH_EDGES, include_lowest=True, right=True
)
if master_population["soh_bin"].isna().any():
    raise AssertionError("Eventos sem bin de SOH")

bin_counts = master_population["soh_bin"].value_counts(sort=False, dropna=False)
if (bin_counts < CV_N_SPLITS).any():
    raise ValueError(f"Bins sem suporte para cinco folds: {bin_counts[bin_counts < CV_N_SPLITS].to_dict()}")

splitter = StratifiedKFold(
    n_splits=CV_N_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE
)
master_population["cv_fold"] = -1
labels = master_population["soh_bin"].astype(str).to_numpy()
for fold_id, (_, validation_indices) in enumerate(
    splitter.split(np.zeros(len(master_population), dtype=np.uint8), labels)
):
    master_population.loc[validation_indices, "cv_fold"] = fold_id
master_population["cv_fold"] = master_population["cv_fold"].astype("int8")

if set(master_population["cv_fold"].unique()) != set(range(CV_N_SPLITS)):
    raise AssertionError("Folds incompletos")
if master_population["event_id"].duplicated().any():
    raise AssertionError("IDs duplicados nas atribuições")

cv_assignments = master_population[["event_id", "soh_bin", "cv_fold"]].copy()
for frame_name in [
    "cv_no_target_cleaning_legacy", "cv_no_target_cleaning_extended",
    "cv_clean1_pooled_legacy", "cv_clean1_pooled_extended",
]:
    frame = globals()[frame_name].merge(
        cv_assignments, on="event_id", how="left", validate="one_to_one", sort=False
    )
    if frame["cv_fold"].isna().any() or frame["soh_bin"].isna().any():
        raise AssertionError(f"Falha ao propagar folds para {frame_name}")
    globals()[frame_name] = frame

shared_clean_ids = set(cv_clean1_pooled_legacy["event_id"])
shared_master = cv_no_target_cleaning_legacy.loc[
    cv_no_target_cleaning_legacy["event_id"].isin(shared_clean_ids), ["event_id"]
].merge(cv_assignments[["event_id", "cv_fold"]], on="event_id", validate="one_to_one")
clean_fold_check = cv_clean1_pooled_legacy[["event_id", "cv_fold"]].merge(
    shared_master, on="event_id", suffixes=("_clean", "_master"), validate="one_to_one"
)
if not clean_fold_check["cv_fold_clean"].eq(clean_fold_check["cv_fold_master"]).all():
    raise AssertionError("Um evento mudou de fold entre os cenários")


In [ ]:
fold_summary_rows = []
for scenario_name, frame in {
    "no_target_cleaning": cv_no_target_cleaning_legacy,
    "clean1_pooled": cv_clean1_pooled_legacy,
}.items():
    for fold_id in range(CV_N_SPLITS):
        validation = frame[frame["cv_fold"].eq(fold_id)]
        training = frame[~frame["cv_fold"].eq(fold_id)]
        fold_summary_rows.append({
            "cenario": scenario_name, "fold": fold_id,
            "treino_eventos": len(training), "validacao_eventos": len(validation),
            "treino_carros": training["car"].nunique(), "validacao_carros": validation["car"].nunique(),
            "carros_compartilhados": len(set(training["car"]) & set(validation["car"])),
            "eventos_compartilhados": len(set(training["event_id"]) & set(validation["event_id"])),
            "validacao_soh_media": validation[TARGET_COL].mean(),
            "validacao_soh_std": validation[TARGET_COL].std(),
        })
fold_summary = pd.DataFrame(fold_summary_rows)

fold_bin_distribution = pd.crosstab(
    master_population["soh_bin"], master_population["cv_fold"], margins=True
)
if not fold_summary["eventos_compartilhados"].eq(0).all():
    raise AssertionError("Vazamento de event_id entre treino e validação")

temporary = CV_ASSIGNMENTS_PATH.with_suffix(".csv.tmp")
cv_assignments.assign(soh_bin=cv_assignments["soh_bin"].astype(str)).to_csv(temporary, index=False)
temporary.replace(CV_ASSIGNMENTS_PATH)
cv_manifest = {
    "stage": "shared_event_folds", "stage_version": CV_STAGE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(), "completed": True,
    "parameters": {
        "method": "StratifiedKFold", "n_splits": CV_N_SPLITS,
        "shuffle": True, "random_state": CV_RANDOM_STATE,
        "soh_min": CV_SOH_MIN, "soh_max": CV_SOH_MAX, "soh_bin_step": CV_SOH_STEP,
        "assignment_population": "no_target_cleaning",
        "generalization_scope": "new_events_from_known_cars",
    },
    "input_sha256": {
        "no_clean": sha256_file(CACHE_DIR / "features_targets_legacy_12128x272.pkl"),
        "clean": sha256_file(CACHE_DIR / "clean1_pooled_legacy.pkl"),
    },
    "shape": list(cv_assignments.shape), "columns": list(cv_assignments.columns),
    "events": len(cv_assignments), "cars": int(master_population["car"].nunique()),
    "fold_counts": {str(k): int(v) for k, v in cv_assignments["cv_fold"].value_counts().sort_index().items()},
    "output_sha256": sha256_file(CV_ASSIGNMENTS_PATH),
    "output_bytes": CV_ASSIGNMENTS_PATH.stat().st_size,
}
temporary_manifest = CV_MANIFEST_PATH.with_suffix(".json.tmp")
temporary_manifest.write_text(json.dumps(cv_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
temporary_manifest.replace(CV_MANIFEST_PATH)

display(fold_summary)
display(fold_bin_distribution)


## 15. Espaços de features para a próxima fase

Serão comparadas duas estratégias dentro de cada fold:

- **Legado fixo:** as 20 features publicadas pelo pipeline original;
- **Seleção train-only:** 20 features escolhidas usando exclusivamente os dados de treino daquele fold.

`mileage` continuará sendo acrescentada após a seleção, resultando em 21 entradas para o modelo. No ramo estendido, as 23 resistências aparentes entram apenas como candidatas; sua permanência dependerá da seleção feita no treino.


In [ ]:
LEGACY_FEATURES_20 = [
    "min_single_volt_slope", "min_single_volt_iqr", "volt_slope", "temp_mean_mean",
    "cell_imbalance_SOC_40_60_std", "cell_imbalance_SOC_40_60_p95",
    "cell_imbalance_SOC_80_100_std", "power_proxy_SOC_60_80_std",
    "temp_mean_SOC_40_60_mean", "min_temp_median", "min_temp_p05",
    "current_abs_p05", "cell_imbalance_SOC_60_80_std", "current_p05",
    "current_abs_SOC_60_80_mean", "current_abs_SOC_40_60_mean",
    "cell_imbalance_SOC_60_80_max", "power_proxy_median", "power_proxy_p05",
    "cell_imbalance_p75",
]
NON_MODEL_SIGNAL_COLUMNS = {"n_snippets", "n_samples", "dt_est", "event_duration_proxy_s"}
legacy_candidate_features = [
    column for column in no_clean_legacy.columns
    if column not in set(META_COLS + LEGACY_CONTEXT_COLUMNS + TOP10_COLUMNS + LABEL_COLUMNS + ["event_id"])
    and column not in NON_MODEL_SIGNAL_COLUMNS
    and pd.api.types.is_numeric_dtype(no_clean_legacy[column])
]
legacy_candidate_features = list(dict.fromkeys(legacy_candidate_features + ["charge_profile_current_code_v1"]))
legacy_candidate_features = [column for column in legacy_candidate_features if column in clean_legacy.columns]
extended_candidate_features = list(dict.fromkeys(legacy_candidate_features + resistance_columns))

missing_legacy_20 = [column for column in LEGACY_FEATURES_20 if column not in legacy_candidate_features]
if missing_legacy_20:
    raise KeyError(f"Features legadas ausentes: {missing_legacy_20}")
if len(LEGACY_FEATURES_20) != 20 or len(set(LEGACY_FEATURES_20)) != 20:
    raise AssertionError("Lista legada não contém 20 features únicas")

feature_space_audit = pd.DataFrame([
    ["features fixas legadas", 20, len(LEGACY_FEATURES_20)],
    ["candidatas legadas", 250, len(legacy_candidate_features)],
    ["features de resistência", 23, len(resistance_columns)],
    ["candidatas estendidas", 273, len(extended_candidate_features)],
    ["mileage acrescentada após seleção", 1, 1],
], columns=["espaço", "esperado", "obtido"])
feature_space_audit["status"] = np.where(
    feature_space_audit["esperado"].eq(feature_space_audit["obtido"]), "OK", "FALHA"
)
display(feature_space_audit)
display(pd.DataFrame({"feature_legada_fixa": LEGACY_FEATURES_20}))


### Resultado da etapa

Os cinco folds por evento foram definidos uma única vez e compartilhados entre os ramos Clean e sem limpeza. A próxima etapa executará a seleção train-only em cada fold, registrará estabilidade e frequência das features e comparará as listas selecionadas com as 20 features fixas do legado.


## 16. Seleção train-only das 20 features

O código legado preservou a lista final das 20 features, mas não um procedimento executável que demonstre como ela foi obtida. Por isso, a lista histórica é tratada como uma configuração fixa de referência.

A seleção nova usa a importância por redução de impureza de um `ExtraTreesRegressor`, escolhida por estar alinhada ao estimador final e capturar relações não lineares. Para cada fold e cenário:

1. somente os eventos dos outros quatro folds são usados;
2. medianas de imputação são calculadas apenas nesse treino;
3. o seletor é ajustado apenas nesse treino;
4. as 20 maiores importâncias são registradas;
5. a validação externa permanece completamente intocada.

A importância por impureza favorece variáveis contínuas e pode repartir importância entre features correlacionadas. Essa limitação será considerada pela análise de estabilidade entre folds; a comparação final de desempenho será feita exclusivamente nos eventos externos de validação.


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

SELECTION_STAGE_VERSION = "feature-selection-v1.0.0-train-only-et"
SELECTION_N_TREES = 300
SELECTION_MIN_SAMPLES_LEAF = 2
SELECTION_RANDOM_STATE = 42
SELECTION_TOP_K = 20
SELECTION_RANKING_PATH = CACHE_DIR / "feature_selection_train_only_rankings.csv"
SELECTION_CONSENSUS_PATH = CACHE_DIR / "feature_selection_train_only_consensus.csv"
SELECTION_MANIFEST_PATH = CACHE_DIR / "feature_selection_train_only.manifest.json"

for frame in (cv_no_target_cleaning_legacy, cv_no_target_cleaning_extended):
    frame["charge_profile_current_code_v1"] = pd.cut(
        frame["current_abs_p95"], [-np.inf, 6, 12, 22, np.inf],
        labels=[0, 1, 2, 3], right=False,
    ).astype("int8")

selection_scenarios = {
    "no_target_cleaning_legacy": (cv_no_target_cleaning_legacy, legacy_candidate_features),
    "no_target_cleaning_extended": (cv_no_target_cleaning_extended, extended_candidate_features),
    "clean1_pooled_legacy": (cv_clean1_pooled_legacy, legacy_candidate_features),
    "clean1_pooled_extended": (cv_clean1_pooled_extended, extended_candidate_features),
}

def fit_train_only_selector(frame, candidate_features, fold_id, scenario_name):
    train = frame.loc[~frame["cv_fold"].eq(fold_id)].copy()
    validation = frame.loc[frame["cv_fold"].eq(fold_id)].copy()
    if set(train["event_id"]) & set(validation["event_id"]):
        raise AssertionError(f"Vazamento de evento: {scenario_name}, fold {fold_id}")

    X_train = train[candidate_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    medians = X_train.median(axis=0)
    all_missing = medians[medians.isna()].index.tolist()
    if all_missing:
        raise ValueError(f"Features integralmente ausentes no treino: {scenario_name}, fold {fold_id}: {all_missing}")
    X_train_imputed = X_train.fillna(medians)
    y_train = pd.to_numeric(train[TARGET_COL], errors="raise").to_numpy(dtype=float)

    selector = ExtraTreesRegressor(
        n_estimators=SELECTION_N_TREES,
        min_samples_leaf=SELECTION_MIN_SAMPLES_LEAF,
        max_features=1.0,
        bootstrap=False,
        random_state=SELECTION_RANDOM_STATE + fold_id,
        n_jobs=-1,
    )
    selector.fit(X_train_imputed, y_train)
    ranking = pd.DataFrame({
        "scenario": scenario_name,
        "fold": fold_id,
        "feature_space": "extended_273" if len(candidate_features) == 273 else "legacy_250",
        "feature": candidate_features,
        "importance": selector.feature_importances_,
        "train_missing_fraction": X_train.isna().mean().to_numpy(),
        "train_imputation_median": medians.to_numpy(),
        "n_train": len(train),
        "n_validation_held_out": len(validation),
    })
    ranking = ranking.sort_values(["importance", "feature"], ascending=[False, True], kind="stable").reset_index(drop=True)
    ranking["rank"] = np.arange(1, len(ranking) + 1)
    ranking["selected_top20"] = ranking["rank"].le(SELECTION_TOP_K)
    if ranking["selected_top20"].sum() != SELECTION_TOP_K:
        raise AssertionError("Quantidade selecionada diferente de 20")
    return ranking


## 17. Execução incremental por cenário e fold

Cada um dos quatro cenários é processado um fold por vez. O ranking inclui a fração ausente e a mediana de imputação de cada feature em cada treino, permitindo auditar as resistências com cobertura limitada.


In [ ]:
def selection_cache_is_compatible():
    required = [SELECTION_RANKING_PATH, SELECTION_CONSENSUS_PATH, SELECTION_MANIFEST_PATH]
    if not all(item.is_file() for item in required):
        return False
    manifest = json.loads(SELECTION_MANIFEST_PATH.read_text(encoding="utf-8"))
    expected_inputs = {
        "cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
        "no_clean_extended": sha256_file(CACHE_DIR / "features_targets_extended_12128x296.pkl"),
        "clean_extended": sha256_file(CACHE_DIR / "clean1_pooled_extended.pkl"),
    }
    return (
        manifest.get("completed") is True
        and manifest.get("stage_version") == SELECTION_STAGE_VERSION
        and manifest.get("input_sha256") == expected_inputs
        and manifest.get("ranking_sha256") == sha256_file(SELECTION_RANKING_PATH)
        and manifest.get("consensus_sha256") == sha256_file(SELECTION_CONSENSUS_PATH)
    )

if selection_cache_is_compatible():
    feature_selection_rankings = pd.read_csv(SELECTION_RANKING_PATH)
    feature_selection_consensus = pd.read_csv(SELECTION_CONSENSUS_PATH)
    feature_selection_rankings["selected_top20"] = feature_selection_rankings["selected_top20"].astype(bool)
    feature_selection_consensus["consensus_top20"] = feature_selection_consensus["consensus_top20"].astype(bool)
    selected_only = feature_selection_rankings[feature_selection_rankings["selected_top20"]].copy()
    print("Cache de seleção: reutilizado")
else:
    selection_rankings = []
    for scenario_name, (scenario_frame, candidate_features) in selection_scenarios.items():
        print(f"Seleção: {scenario_name}")
        for fold_id in range(CV_N_SPLITS):
            ranking = fit_train_only_selector(scenario_frame, candidate_features, fold_id, scenario_name)
            selection_rankings.append(ranking)
            selected_names = ranking.loc[ranking["selected_top20"], "feature"].tolist()
            print(
                f"  fold {fold_id}: n_train={int(ranking['n_train'].iloc[0])}; "
                f"resistências={sum(name.startswith('apparent_resistance_') for name in selected_names)}"
            )

    feature_selection_rankings = pd.concat(selection_rankings, ignore_index=True)
    expected_ranking_rows = CV_N_SPLITS * (2 * 250 + 2 * 273)
    if len(feature_selection_rankings) != expected_ranking_rows:
        raise AssertionError(f"Linhas de ranking: {len(feature_selection_rankings)} != {expected_ranking_rows}")

    selected_only = feature_selection_rankings[feature_selection_rankings["selected_top20"]].copy()
    if len(selected_only) != len(selection_scenarios) * CV_N_SPLITS * SELECTION_TOP_K:
        raise AssertionError("Número total de seleções inconsistente")

    consensus_rows = []
    for scenario_name, group in feature_selection_rankings.groupby("scenario", sort=True):
        consensus = group.groupby("feature", as_index=False).agg(
            folds_selected=("selected_top20", "sum"),
            mean_rank=("rank", "mean"),
            rank_std=("rank", "std"),
            mean_importance=("importance", "mean"),
            importance_std=("importance", "std"),
            mean_missing_fraction=("train_missing_fraction", "mean"),
        )
        consensus.insert(0, "scenario", scenario_name)
        consensus = consensus.sort_values(
            ["folds_selected", "mean_rank", "mean_importance", "feature"],
            ascending=[False, True, False, True], kind="stable",
        ).reset_index(drop=True)
        consensus["consensus_rank"] = np.arange(1, len(consensus) + 1)
        consensus["consensus_top20"] = consensus["consensus_rank"].le(SELECTION_TOP_K)
        consensus_rows.append(consensus)
    feature_selection_consensus = pd.concat(consensus_rows, ignore_index=True)

    temporary_rankings = SELECTION_RANKING_PATH.with_suffix(".csv.tmp")
    feature_selection_rankings.to_csv(temporary_rankings, index=False)
    temporary_rankings.replace(SELECTION_RANKING_PATH)
    temporary_consensus = SELECTION_CONSENSUS_PATH.with_suffix(".csv.tmp")
    feature_selection_consensus.to_csv(temporary_consensus, index=False)
    temporary_consensus.replace(SELECTION_CONSENSUS_PATH)
    print("Cache de seleção: construído")


## 18. Estabilidade e comparação com as 20 features legadas

A estabilidade é descrita pela frequência de seleção em cinco folds e pelo índice de Jaccard entre pares de listas. O consenso não usa a validação: ordena primeiro pela quantidade de folds em que a feature entrou no Top-20, depois pelo rank médio e pela importância média.


In [ ]:
def pairwise_jaccard(selected_frame):
    rows = []
    for scenario_name, scenario_group in selected_frame.groupby("scenario", sort=True):
        feature_sets = {
            int(fold): set(group["feature"])
            for fold, group in scenario_group.groupby("fold", sort=True)
        }
        for fold_a in range(CV_N_SPLITS):
            for fold_b in range(fold_a + 1, CV_N_SPLITS):
                set_a, set_b = feature_sets[fold_a], feature_sets[fold_b]
                rows.append({
                    "scenario": scenario_name, "fold_a": fold_a, "fold_b": fold_b,
                    "intersection": len(set_a & set_b), "union": len(set_a | set_b),
                    "jaccard": len(set_a & set_b) / len(set_a | set_b),
                })
    return pd.DataFrame(rows)

selection_jaccard = pairwise_jaccard(selected_only)
scenario_summary_rows = []
legacy_set = set(LEGACY_FEATURES_20)
for scenario_name, scenario_consensus in feature_selection_consensus.groupby("scenario", sort=True):
    consensus_set = set(scenario_consensus.loc[scenario_consensus["consensus_top20"], "feature"])
    scenario_jaccard = selection_jaccard[selection_jaccard["scenario"].eq(scenario_name)]
    scenario_summary_rows.append({
        "scenario": scenario_name,
        "candidate_features": int((feature_selection_rankings["scenario"] == scenario_name).groupby(feature_selection_rankings["fold"]).sum().iloc[0]),
        "stable_in_5_folds": int((scenario_consensus["folds_selected"] == 5).sum()),
        "mean_pairwise_jaccard": scenario_jaccard["jaccard"].mean(),
        "min_pairwise_jaccard": scenario_jaccard["jaccard"].min(),
        "overlap_with_legacy_20": len(consensus_set & legacy_set),
        "jaccard_with_legacy_20": len(consensus_set & legacy_set) / len(consensus_set | legacy_set),
        "resistance_in_consensus_top20": sum(name.startswith("apparent_resistance_") for name in consensus_set),
    })
feature_selection_summary = pd.DataFrame(scenario_summary_rows)

consensus_top20_table = feature_selection_consensus.loc[
    feature_selection_consensus["consensus_top20"],
    ["scenario", "consensus_rank", "feature", "folds_selected", "mean_rank", "mean_importance", "mean_missing_fraction"],
].copy()

selection_manifest = {
    "stage": "train_only_feature_selection", "stage_version": SELECTION_STAGE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(), "completed": True,
    "parameters": {
        "selector": "ExtraTreesRegressor_MDI", "top_k": SELECTION_TOP_K,
        "n_estimators": SELECTION_N_TREES, "min_samples_leaf": SELECTION_MIN_SAMPLES_LEAF,
        "max_features": 1.0, "bootstrap": False, "random_state_base": SELECTION_RANDOM_STATE,
        "imputation": "per_feature_training_median", "validation_used_for_selection": False,
        "scenarios": list(selection_scenarios),
    },
    "input_sha256": {
        "cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
        "no_clean_extended": sha256_file(CACHE_DIR / "features_targets_extended_12128x296.pkl"),
        "clean_extended": sha256_file(CACHE_DIR / "clean1_pooled_extended.pkl"),
    },
    "ranking_shape": list(feature_selection_rankings.shape),
    "consensus_shape": list(feature_selection_consensus.shape),
    "ranking_sha256": sha256_file(SELECTION_RANKING_PATH),
    "consensus_sha256": sha256_file(SELECTION_CONSENSUS_PATH),
    "selected_per_scenario_fold": SELECTION_TOP_K,
}
temporary_manifest = SELECTION_MANIFEST_PATH.with_suffix(".json.tmp")
temporary_manifest.write_text(json.dumps(selection_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
temporary_manifest.replace(SELECTION_MANIFEST_PATH)

display(feature_selection_summary)
display(consensus_top20_table)
display(selection_jaccard.groupby("scenario")["jaccard"].describe())


### Resultado da etapa

As listas por fold, rankings completos, medianas de imputação e consensos foram salvos em cache. A próxima etapa comparará o desempenho externo das features fixas legadas e das features train-only, mantendo o mesmo estimador, os mesmos folds e o mesmo tratamento de dados em cada cenário.


## 19. Avaliação externa comparativa sem SMOTER

Esta etapa isola três fatores: limpeza, seleção de features e inclusão das resistências aparentes. O SMOTER ainda não é aplicado, pois introduzir balanceamento agora impediria atribuir diferenças de desempenho a um único componente.

Para cada população (`no_target_cleaning` e `clean1_pooled`) são comparados:

1. `legacy_fixed_20`: as 20 features históricas fixas;
2. `train_only_legacy_20`: as 20 selecionadas no treino entre 250 candidatas;
3. `train_only_extended_20`: as 20 selecionadas no treino entre 273 candidatas.

`mileage` é acrescentada às três listas, resultando em 21 entradas. A população sem limpeza é a análise principal. A Clean 1 pooled é exploratória, pois sua filtragem baseada no target foi calculada antes dos folds.


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

EVALUATION_STAGE_VERSION = "evaluation-v1.0.0-no-smoter"
MODEL_PARAMETERS = {
    "n_estimators": 1200,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "bootstrap": False,
    "random_state": 42,
    "n_jobs": -1,
}
EVALUATION_METRICS_PATH = CACHE_DIR / "evaluation_no_smoter_metrics.csv"
EVALUATION_PREDICTIONS_PATH = CACHE_DIR / "evaluation_no_smoter_oof_predictions.csv"
EVALUATION_FEATURES_PATH = CACHE_DIR / "evaluation_no_smoter_feature_lists.json"
EVALUATION_MANIFEST_PATH = CACHE_DIR / "evaluation_no_smoter.manifest.json"

evaluation_input_hashes = {
    "cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
    "feature_rankings": sha256_file(SELECTION_RANKING_PATH),
    "no_clean_legacy": sha256_file(CACHE_DIR / "features_targets_legacy_12128x272.pkl"),
    "no_clean_extended": sha256_file(CACHE_DIR / "features_targets_extended_12128x296.pkl"),
    "clean_legacy": sha256_file(CACHE_DIR / "clean1_pooled_legacy.pkl"),
    "clean_extended": sha256_file(CACHE_DIR / "clean1_pooled_extended.pkl"),
}

evaluation_populations = {
    "no_target_cleaning": {
        "legacy": cv_no_target_cleaning_legacy,
        "extended": cv_no_target_cleaning_extended,
        "ranking_legacy": "no_target_cleaning_legacy",
        "ranking_extended": "no_target_cleaning_extended",
    },
    "clean1_pooled": {
        "legacy": cv_clean1_pooled_legacy,
        "extended": cv_clean1_pooled_extended,
        "ranking_legacy": "clean1_pooled_legacy",
        "ranking_extended": "clean1_pooled_extended",
    },
}

def selected_features_for_fold(ranking_scenario, fold_id):
    selected = feature_selection_rankings.loc[
        feature_selection_rankings["scenario"].eq(ranking_scenario)
        & feature_selection_rankings["fold"].eq(fold_id)
        & feature_selection_rankings["selected_top20"],
    ].sort_values("rank")["feature"].tolist()
    if len(selected) != 20 or len(set(selected)) != 20:
        raise AssertionError(f"Lista train-only inválida: {ranking_scenario}, fold {fold_id}")
    return selected

def regression_metrics(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias": float(error.mean()),
        "error_std": float(error.std(ddof=1)),
    }

def evaluation_cache_is_compatible():
    required = [EVALUATION_METRICS_PATH, EVALUATION_PREDICTIONS_PATH, EVALUATION_FEATURES_PATH, EVALUATION_MANIFEST_PATH]
    if not all(item.is_file() for item in required):
        return False
    manifest = json.loads(EVALUATION_MANIFEST_PATH.read_text(encoding="utf-8"))
    return (
        manifest.get("completed") is True
        and manifest.get("stage_version") == EVALUATION_STAGE_VERSION
        and manifest.get("parameters") == MODEL_PARAMETERS
        and manifest.get("input_sha256") == evaluation_input_hashes
        and manifest.get("metrics_sha256") == sha256_file(EVALUATION_METRICS_PATH)
        and manifest.get("predictions_sha256") == sha256_file(EVALUATION_PREDICTIONS_PATH)
        and manifest.get("feature_lists_sha256") == sha256_file(EVALUATION_FEATURES_PATH)
    )


## 20. Treinamento incremental e predições OOF

Cada modelo é treinado e descartado antes do próximo. A imputação por mediana é ajustada somente no treino do fold e aplicada à validação. As listas train-only são específicas do fold; o consenso global não é usado para predizer.


In [ ]:
if evaluation_cache_is_compatible():
    evaluation_metrics_by_fold = pd.read_csv(EVALUATION_METRICS_PATH)
    evaluation_oof_predictions = pd.read_csv(EVALUATION_PREDICTIONS_PATH)
    evaluation_feature_lists = json.loads(EVALUATION_FEATURES_PATH.read_text(encoding="utf-8"))
    evaluation_cache_status = "reutilizado"
else:
    metric_rows = []
    prediction_frames = []
    evaluation_feature_lists = {}
    total_start = time.perf_counter()

    for population_name, population in evaluation_populations.items():
        print(f"População: {population_name}")
        method_definitions = {
            "legacy_fixed_20": (population["legacy"], LEGACY_FEATURES_20),
            "train_only_legacy_20": (population["legacy"], None),
            "train_only_extended_20": (population["extended"], None),
        }
        for fold_id in range(CV_N_SPLITS):
            for method_name, (frame, fixed_features) in method_definitions.items():
                if method_name == "train_only_legacy_20":
                    selected_20 = selected_features_for_fold(population["ranking_legacy"], fold_id)
                elif method_name == "train_only_extended_20":
                    selected_20 = selected_features_for_fold(population["ranking_extended"], fold_id)
                else:
                    selected_20 = list(fixed_features)
                model_features = list(dict.fromkeys(selected_20 + ["mileage"]))
                if len(model_features) != 21:
                    raise AssertionError(f"Esperadas 21 entradas: {population_name}, {method_name}, fold {fold_id}")

                train = frame.loc[~frame["cv_fold"].eq(fold_id)].copy()
                validation = frame.loc[frame["cv_fold"].eq(fold_id)].copy()
                if set(train["event_id"]) & set(validation["event_id"]):
                    raise AssertionError("Vazamento de event_id")

                X_train = train[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
                X_validation = validation[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
                medians = X_train.median(axis=0)
                if medians.isna().any():
                    raise ValueError(f"Mediana ausente: {medians[medians.isna()].index.tolist()}")
                X_train = X_train.fillna(medians)
                X_validation = X_validation.fillna(medians)
                y_train = train[TARGET_COL].to_numpy(dtype=float)
                y_validation = validation[TARGET_COL].to_numpy(dtype=float)

                start = time.perf_counter()
                model = ExtraTreesRegressor(**MODEL_PARAMETERS)
                model.fit(X_train, y_train)
                predictions = model.predict(X_validation)
                elapsed = time.perf_counter() - start
                metrics = regression_metrics(y_validation, predictions)
                metric_rows.append({
                    "population": population_name, "feature_method": method_name, "fold": fold_id,
                    "n_train": len(train), "n_validation": len(validation), "n_features": len(model_features),
                    "n_resistance_features": sum(name.startswith("apparent_resistance_") for name in model_features),
                    "fit_predict_seconds": elapsed, **metrics,
                })
                prediction_frames.append(pd.DataFrame({
                    "population": population_name, "feature_method": method_name, "fold": fold_id,
                    "event_id": validation["event_id"].to_numpy(), "car": validation["car"].to_numpy(),
                    "mileage": validation["mileage"].to_numpy(), "soh_real": y_validation,
                    "soh_pred": predictions, "error": predictions-y_validation,
                    "abs_error": np.abs(predictions-y_validation),
                }))
                list_key = f"{population_name}|{method_name}|fold_{fold_id}"
                evaluation_feature_lists[list_key] = {
                    "selected_20": selected_20, "model_features_21": model_features,
                    "training_medians": {key: float(value) for key, value in medians.items()},
                }
                print(
                    f"  fold {fold_id} | {method_name}: RMSE={metrics['rmse']:.6f}; "
                    f"MAE={metrics['mae']:.6f}; {elapsed:.1f}s"
                )
                del model, X_train, X_validation, train, validation

    evaluation_metrics_by_fold = pd.DataFrame(metric_rows)
    evaluation_oof_predictions = pd.concat(prediction_frames, ignore_index=True)
    temporary_metrics = EVALUATION_METRICS_PATH.with_suffix(".csv.tmp")
    evaluation_metrics_by_fold.to_csv(temporary_metrics, index=False)
    temporary_metrics.replace(EVALUATION_METRICS_PATH)
    temporary_predictions = EVALUATION_PREDICTIONS_PATH.with_suffix(".csv.tmp")
    evaluation_oof_predictions.to_csv(temporary_predictions, index=False)
    temporary_predictions.replace(EVALUATION_PREDICTIONS_PATH)
    temporary_features = EVALUATION_FEATURES_PATH.with_suffix(".json.tmp")
    temporary_features.write_text(json.dumps(evaluation_feature_lists, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_features.replace(EVALUATION_FEATURES_PATH)
    evaluation_manifest = {
        "stage": "external_comparison_no_smoter", "stage_version": EVALUATION_STAGE_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "completed": True,
        "parameters": MODEL_PARAMETERS, "input_sha256": evaluation_input_hashes,
        "metrics_shape": list(evaluation_metrics_by_fold.shape),
        "predictions_shape": list(evaluation_oof_predictions.shape),
        "metrics_sha256": sha256_file(EVALUATION_METRICS_PATH),
        "predictions_sha256": sha256_file(EVALUATION_PREDICTIONS_PATH),
        "feature_lists_sha256": sha256_file(EVALUATION_FEATURES_PATH),
        "elapsed_seconds": time.perf_counter() - total_start,
    }
    temporary_manifest = EVALUATION_MANIFEST_PATH.with_suffix(".json.tmp")
    temporary_manifest.write_text(json.dumps(evaluation_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_manifest.replace(EVALUATION_MANIFEST_PATH)
    evaluation_cache_status = "construído"

print("Cache de avaliação:", evaluation_cache_status)


## 21. Métricas globais e comparação pareada por carro

As métricas OOF agregam exatamente uma predição externa por evento e método. Para comparar RMSE, usamos bootstrap pareado por carro: carros inteiros são reamostrados com reposição, preservando a dependência entre eventos do mesmo veículo. O intervalo é descritivo e não corrige a filtragem retrospectiva da Clean 1 pooled.


In [ ]:
expected_fold_rows = len(evaluation_populations) * 3 * CV_N_SPLITS
if len(evaluation_metrics_by_fold) != expected_fold_rows:
    raise AssertionError(f"Métricas por fold: {len(evaluation_metrics_by_fold)} != {expected_fold_rows}")
if evaluation_oof_predictions.duplicated(["population", "feature_method", "event_id"]).any():
    raise AssertionError("Predições OOF duplicadas")

global_metric_rows = []
for (population_name, method_name), group in evaluation_oof_predictions.groupby(
    ["population", "feature_method"], sort=True
):
    global_metric_rows.append({
        "population": population_name, "feature_method": method_name,
        "n_events": len(group), **regression_metrics(group["soh_real"], group["soh_pred"]),
    })
evaluation_global_metrics = pd.DataFrame(global_metric_rows)

def paired_car_bootstrap_delta_rmse(predictions, population_name, comparison_method, baseline_method="legacy_fixed_20", n_bootstrap=2000, seed=42):
    subset = predictions.loc[
        predictions["population"].eq(population_name)
        & predictions["feature_method"].isin([baseline_method, comparison_method]),
        ["feature_method", "event_id", "car", "soh_real", "soh_pred"],
    ]
    wide = subset.pivot(index=["event_id", "car", "soh_real"], columns="feature_method", values="soh_pred").reset_index()
    if wide[[baseline_method, comparison_method]].isna().any().any():
        raise AssertionError("Comparação pareada incompleta")
    cars = wide["car"].drop_duplicates().to_numpy()
    groups = {car: wide.index[wide["car"].eq(car)].to_numpy() for car in cars}
    rng = np.random.default_rng(seed)
    deltas = np.empty(n_bootstrap, dtype=float)
    for iteration in range(n_bootstrap):
        sampled_cars = rng.choice(cars, size=len(cars), replace=True)
        indices = np.concatenate([groups[car] for car in sampled_cars])
        y = wide.loc[indices, "soh_real"].to_numpy()
        rmse_fixed = np.sqrt(np.mean((wide.loc[indices, baseline_method].to_numpy() - y) ** 2))
        rmse_comparison = np.sqrt(np.mean((wide.loc[indices, comparison_method].to_numpy() - y) ** 2))
        deltas[iteration] = rmse_comparison - rmse_fixed
    observed_fixed = np.sqrt(np.mean((wide[baseline_method] - wide["soh_real"]) ** 2))
    observed_comparison = np.sqrt(np.mean((wide[comparison_method] - wide["soh_real"]) ** 2))
    return {
        "population": population_name, "comparison": f"{comparison_method} - {baseline_method}",
        "n_events": len(wide), "n_cars": len(cars),
        "rmse_fixed": observed_fixed, "rmse_comparison": observed_comparison,
        "delta_rmse": observed_comparison-observed_fixed,
        "delta_ci95_low": np.quantile(deltas, 0.025),
        "delta_ci95_high": np.quantile(deltas, 0.975),
        "bootstrap_probability_delta_below_zero": np.mean(deltas < 0),
    }

bootstrap_comparisons = pd.DataFrame([
    paired_car_bootstrap_delta_rmse(
        evaluation_oof_predictions, population, comparison_method, baseline_method
    )
    for population in evaluation_populations
    for baseline_method, comparison_method in [
        ("legacy_fixed_20", "train_only_legacy_20"),
        ("legacy_fixed_20", "train_only_extended_20"),
        ("train_only_legacy_20", "train_only_extended_20"),
    ]
])

display(evaluation_metrics_by_fold)
display(evaluation_global_metrics.sort_values(["population", "rmse"]))
display(bootstrap_comparisons)


### Resultado da etapa

A comparação externa foi executada sem SMOTER, mantendo constantes folds, estimador e número de entradas. Na próxima etapa, os resultados indicarão qual configuração deve seguir para o balanceamento e para as análises de incerteza; qualquer ganho será interpretado em conjunto com o intervalo bootstrap por carro.


## 22. SMOTER condicionado somente no treino

O balanceamento segue a versão auditável do método legado:

- apenas eventos com `mileage ≥ 30.000` podem ser pais;
- os dois pais pertencem ao mesmo bin de SOH;
- a diferença de mileage entre pais é no máximo 15.000;
- são considerados até cinco vizinhos mais próximos;
- o primeiro bin tem alvo de 300 eventos e os demais, 500;
- features, SOH e mileage sintéticos são interpolados entre os dois pais.

As distâncias usam as 250 features físicas legadas, com `RobustScaler` e medianas ajustados somente nos elegíveis do treino. O modelo continua recebendo apenas as 20 features fixas mais `mileage`. A validação contém exclusivamente eventos reais.


In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import RobustScaler

SMOTER_STAGE_VERSION = "smoter-v1.0.0-conditioned-train-only"
SMOTER_MILEAGE_MIN = 30_000
SMOTER_MAX_MILEAGE_DIFF = 15_000
SMOTER_K_NEIGHBORS = 5
SMOTER_TARGET_FIRST_BIN = 300
SMOTER_TARGET_OTHER_BINS = 500
SMOTER_RANDOM_STATE = 42

SMOTER_METRICS_PATH = CACHE_DIR / "smoter_legacy_fixed_metrics.csv"
SMOTER_PREDICTIONS_PATH = CACHE_DIR / "smoter_legacy_fixed_oof_predictions.csv"
SMOTER_BALANCE_PATH = CACHE_DIR / "smoter_balance_plan.csv"
SMOTER_GENERATION_PATH = CACHE_DIR / "smoter_generation_audit.csv"
SMOTER_NEIGHBOR_PATH = CACHE_DIR / "smoter_neighbor_summary.csv"
SMOTER_BIN_METRICS_PATH = CACHE_DIR / "smoter_metrics_by_soh_bin.csv"
SMOTER_MANIFEST_PATH = CACHE_DIR / "smoter_legacy_fixed.manifest.json"

smoter_input_hashes = {
    "cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
    "no_clean": sha256_file(CACHE_DIR / "features_targets_legacy_12128x272.pkl"),
    "clean": sha256_file(CACHE_DIR / "clean1_pooled_legacy.pkl"),
    "baseline_predictions": sha256_file(EVALUATION_PREDICTIONS_PATH),
}

def smoter_target_for_bin(interval):
    right = float(interval.right)
    tolerance = 1e-8
    if right <= 0.925 + tolerance and right > 0.920:
        return SMOTER_TARGET_FIRST_BIN
    if right > 0.925 + tolerance and right <= CV_SOH_MAX + tolerance:
        return SMOTER_TARGET_OTHER_BINS
    return None

def build_smoter_training(train_real, population_name, fold_id, distance_features):
    rng = np.random.default_rng(SMOTER_RANDOM_STATE + int(fold_id))
    train_real = train_real.copy().reset_index(drop=True)
    train_real["sample_origin"] = "real"
    eligible = train_real.loc[train_real["mileage"] >= SMOTER_MILEAGE_MIN].copy().reset_index(drop=True)

    X_eligible = eligible[distance_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    feature_medians = X_eligible.median(axis=0)
    if feature_medians.isna().any():
        raise ValueError(f"Features sem valores elegíveis: {feature_medians[feature_medians.isna()].index.tolist()}")
    X_imputed = X_eligible.fillna(feature_medians)
    variable_features = X_imputed.columns[X_imputed.nunique(dropna=False).gt(1)].tolist()
    if not variable_features:
        raise ValueError("Nenhuma feature variável para distâncias SMOTER")
    scaler = RobustScaler()
    X_scaled = pd.DataFrame(
        scaler.fit_transform(X_imputed[variable_features]),
        columns=variable_features, index=eligible.index,
    )
    X_original = X_imputed.reset_index(drop=True)

    neighbor_map = {}
    neighbor_summary_rows = []
    for soh_bin, bin_frame in eligible.groupby("soh_bin", observed=True):
        indices = bin_frame.index.to_numpy()
        if len(indices) < 2:
            for index in indices:
                neighbor_map[int(index)] = []
            neighbor_summary_rows.append({
                "population": population_name, "fold": fold_id, "soh_bin": str(soh_bin),
                "eligible_events": len(indices), "events_with_valid_neighbor": 0,
                "mean_neighbors_available": 0.0,
            })
            continue
        distances = pairwise_distances(X_scaled.loc[indices].to_numpy(), metric="euclidean")
        mileages = bin_frame["mileage"].to_numpy(dtype=float)
        allowed = np.abs(mileages[:, None] - mileages[None, :]) <= SMOTER_MAX_MILEAGE_DIFF
        np.fill_diagonal(allowed, False)
        distances[~allowed] = np.inf
        neighbor_counts = []
        for position, global_index in enumerate(indices):
            valid_positions = np.flatnonzero(np.isfinite(distances[position]))
            ordered = valid_positions[np.argsort(distances[position, valid_positions])]
            neighbors = indices[ordered[:SMOTER_K_NEIGHBORS]].astype(int).tolist()
            neighbor_map[int(global_index)] = neighbors
            neighbor_counts.append(len(neighbors))
        neighbor_summary_rows.append({
            "population": population_name, "fold": fold_id, "soh_bin": str(soh_bin),
            "eligible_events": len(indices),
            "events_with_valid_neighbor": int(np.sum(np.asarray(neighbor_counts) > 0)),
            "mean_neighbors_available": float(np.mean(neighbor_counts)),
        })

    train_bin_counts = train_real.groupby("soh_bin", observed=True).size()
    balance_rows = []
    valid_seeds_by_bin = {}
    for soh_bin, n_real in train_bin_counts.items():
        target = smoter_target_for_bin(soh_bin)
        eligible_indices = eligible.index[eligible["soh_bin"].eq(soh_bin)].tolist()
        valid_seeds = [int(index) for index in eligible_indices if neighbor_map.get(int(index), [])]
        valid_seeds_by_bin[soh_bin] = valid_seeds
        requested = 0 if target is None or n_real >= target else int(target - n_real)
        if requested and not valid_seeds:
            raise ValueError(f"Fold {fold_id}, bin {soh_bin}: geração solicitada sem pais válidos")
        balance_rows.append({
            "population": population_name, "fold": fold_id, "soh_bin": str(soh_bin),
            "n_real_train": int(n_real), "target_final": target,
            "n_synthetic_requested": requested, "n_eligible_mileage": len(eligible_indices),
            "n_valid_seeds": len(valid_seeds),
        })

    synthetic_rows = []
    generation_rows = []
    counter = 0
    for balance_row, (soh_bin, _) in zip(balance_rows, train_bin_counts.items()):
        for _ in range(balance_row["n_synthetic_requested"]):
            seed_index = int(rng.choice(valid_seeds_by_bin[soh_bin]))
            neighbor_index = int(rng.choice(neighbor_map[seed_index]))
            alpha = float(rng.random())
            parent_a = eligible.loc[seed_index]
            parent_b = eligible.loc[neighbor_index]
            synthetic_features = X_original.loc[seed_index] + alpha * (X_original.loc[neighbor_index] - X_original.loc[seed_index])
            counter += 1
            row = parent_a.to_dict()
            for feature in distance_features:
                row[feature] = float(synthetic_features[feature])
            row["event_id"] = f"smoter_{population_name}_f{fold_id}_{counter:06d}"
            row[TARGET_COL] = float(parent_a[TARGET_COL] + alpha * (parent_b[TARGET_COL] - parent_a[TARGET_COL]))
            row["mileage"] = float(parent_a["mileage"] + alpha * (parent_b["mileage"] - parent_a["mileage"]))
            row["soh_bin"] = soh_bin
            row["sample_origin"] = "smoter"
            synthetic_rows.append(row)
            generation_rows.append({
                "population": population_name, "fold": fold_id, "synthetic_event_id": row["event_id"],
                "soh_bin": str(soh_bin), "parent_a_event_id": parent_a["event_id"],
                "parent_b_event_id": parent_b["event_id"], "parent_a_soh": float(parent_a[TARGET_COL]),
                "parent_b_soh": float(parent_b[TARGET_COL]), "synthetic_soh": row[TARGET_COL],
                "parent_a_mileage": float(parent_a["mileage"]), "parent_b_mileage": float(parent_b["mileage"]),
                "synthetic_mileage": row["mileage"],
                "parent_mileage_diff": float(abs(parent_a["mileage"] - parent_b["mileage"])),
                "interpolation_factor": alpha,
            })
    synthetic = pd.DataFrame(synthetic_rows)
    if len(synthetic):
        balanced = pd.concat([train_real, synthetic[train_real.columns]], ignore_index=True)
    else:
        balanced = train_real.copy()
    if len(synthetic) != sum(row["n_synthetic_requested"] for row in balance_rows):
        raise AssertionError("Quantidade sintética diferente do plano")
    if generation_rows and max(row["parent_mileage_diff"] for row in generation_rows) > SMOTER_MAX_MILEAGE_DIFF + 1e-9:
        raise AssertionError("Pais acima da distância máxima de mileage")
    return balanced, pd.DataFrame(balance_rows), pd.DataFrame(generation_rows), pd.DataFrame(neighbor_summary_rows)


## 23. Treinamento balanceado e validação real

São avaliadas as populações sem limpeza e Clean 1 pooled. Cada validação mantém exatamente os mesmos eventos do baseline sem SMOTER, permitindo comparação pareada.


In [ ]:
SMOTER_REQUIRED_CACHE = [
    SMOTER_METRICS_PATH, SMOTER_PREDICTIONS_PATH, SMOTER_BALANCE_PATH,
    SMOTER_GENERATION_PATH, SMOTER_NEIGHBOR_PATH, SMOTER_BIN_METRICS_PATH, SMOTER_MANIFEST_PATH,
]

def smoter_cache_is_compatible():
    if not all(path.is_file() for path in SMOTER_REQUIRED_CACHE):
        return False
    manifest = json.loads(SMOTER_MANIFEST_PATH.read_text(encoding="utf-8"))
    return (
        manifest.get("completed") is True
        and manifest.get("stage_version") == SMOTER_STAGE_VERSION
        and manifest.get("input_sha256") == smoter_input_hashes
        and manifest.get("model_parameters") == MODEL_PARAMETERS
        and manifest.get("predictions_sha256") == sha256_file(SMOTER_PREDICTIONS_PATH)
    )

if smoter_cache_is_compatible():
    smoter_metrics_by_fold = pd.read_csv(SMOTER_METRICS_PATH)
    smoter_oof_predictions = pd.read_csv(SMOTER_PREDICTIONS_PATH)
    smoter_balance_plan = pd.read_csv(SMOTER_BALANCE_PATH)
    smoter_generation_audit = pd.read_csv(SMOTER_GENERATION_PATH)
    smoter_neighbor_summary = pd.read_csv(SMOTER_NEIGHBOR_PATH)
    smoter_metrics_by_bin = pd.read_csv(SMOTER_BIN_METRICS_PATH)
    smoter_cache_status = "reutilizado"
else:
    metric_rows, prediction_frames = [], []
    balance_frames, generation_frames, neighbor_frames = [], [], []
    start_total = time.perf_counter()
    smoter_populations = {
        "no_target_cleaning": cv_no_target_cleaning_legacy,
        "clean1_pooled": cv_clean1_pooled_legacy,
    }
    model_features = LEGACY_FEATURES_20 + ["mileage"]
    for population_name, frame in smoter_populations.items():
        print(f"SMOTER: {population_name}")
        for fold_id in range(CV_N_SPLITS):
            train_real = frame.loc[~frame["cv_fold"].eq(fold_id)].copy()
            validation = frame.loc[frame["cv_fold"].eq(fold_id)].copy()
            balanced, balance, generation, neighbor = build_smoter_training(
                train_real, population_name, fold_id, legacy_candidate_features
            )
            balance_frames.append(balance); generation_frames.append(generation); neighbor_frames.append(neighbor)
            X_train = balanced[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
            X_valid = validation[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
            medians = X_train.median()
            X_train = X_train.fillna(medians); X_valid = X_valid.fillna(medians)
            y_train = balanced[TARGET_COL].to_numpy(dtype=float)
            y_valid = validation[TARGET_COL].to_numpy(dtype=float)
            start = time.perf_counter()
            model = ExtraTreesRegressor(**MODEL_PARAMETERS)
            model.fit(X_train, y_train)
            predictions = model.predict(X_valid)
            elapsed = time.perf_counter() - start
            metrics = regression_metrics(y_valid, predictions)
            n_synthetic = int(balanced["sample_origin"].eq("smoter").sum())
            metric_rows.append({
                "population": population_name, "fold": fold_id, "n_train_real": len(train_real),
                "n_train_synthetic": n_synthetic, "n_train_balanced": len(balanced),
                "n_validation_real": len(validation), "fit_predict_seconds": elapsed, **metrics,
            })
            prediction_frames.append(pd.DataFrame({
                "population": population_name, "feature_method": "legacy_fixed_20_smoter",
                "fold": fold_id, "event_id": validation["event_id"].to_numpy(),
                "car": validation["car"].to_numpy(), "mileage": validation["mileage"].to_numpy(),
                "soh_bin": validation["soh_bin"].astype(str).to_numpy(), "soh_real": y_valid,
                "soh_pred": predictions, "error": predictions-y_valid,
                "abs_error": np.abs(predictions-y_valid),
            }))
            print(f"  fold {fold_id}: reais={len(train_real)}, sintéticos={n_synthetic}, RMSE={metrics['rmse']:.6f}")
            del model, balanced, X_train, X_valid, train_real, validation

    smoter_metrics_by_fold = pd.DataFrame(metric_rows)
    smoter_oof_predictions = pd.concat(prediction_frames, ignore_index=True)
    smoter_balance_plan = pd.concat(balance_frames, ignore_index=True)
    smoter_generation_audit = pd.concat(generation_frames, ignore_index=True)
    smoter_neighbor_summary = pd.concat(neighbor_frames, ignore_index=True)
    bin_rows = []
    for (population, soh_bin), group in smoter_oof_predictions.groupby(["population", "soh_bin"], sort=True):
        bin_rows.append({"population": population, "soh_bin": soh_bin, "n_events": len(group), **regression_metrics(group["soh_real"], group["soh_pred"])})
    smoter_metrics_by_bin = pd.DataFrame(bin_rows)

    for frame_to_save, output_path in [
        (smoter_metrics_by_fold, SMOTER_METRICS_PATH), (smoter_oof_predictions, SMOTER_PREDICTIONS_PATH),
        (smoter_balance_plan, SMOTER_BALANCE_PATH), (smoter_generation_audit, SMOTER_GENERATION_PATH),
        (smoter_neighbor_summary, SMOTER_NEIGHBOR_PATH), (smoter_metrics_by_bin, SMOTER_BIN_METRICS_PATH),
    ]:
        temporary = output_path.with_suffix(".csv.tmp")
        frame_to_save.to_csv(temporary, index=False); temporary.replace(output_path)
    manifest = {
        "stage": "smoter_legacy_fixed_external_evaluation", "stage_version": SMOTER_STAGE_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "completed": True,
        "parameters": {
            "mileage_min": SMOTER_MILEAGE_MIN, "max_parent_mileage_diff": SMOTER_MAX_MILEAGE_DIFF,
            "k_neighbors": SMOTER_K_NEIGHBORS, "target_first_bin": SMOTER_TARGET_FIRST_BIN,
            "target_other_bins": SMOTER_TARGET_OTHER_BINS, "random_state": SMOTER_RANDOM_STATE,
            "distance_features": len(legacy_candidate_features), "model_features": LEGACY_FEATURES_20 + ["mileage"],
        },
        "model_parameters": MODEL_PARAMETERS, "input_sha256": smoter_input_hashes,
        "metrics_shape": list(smoter_metrics_by_fold.shape), "predictions_shape": list(smoter_oof_predictions.shape),
        "synthetic_events_generated": int(len(smoter_generation_audit)),
        "predictions_sha256": sha256_file(SMOTER_PREDICTIONS_PATH),
        "elapsed_seconds": time.perf_counter()-start_total,
    }
    temporary_manifest = SMOTER_MANIFEST_PATH.with_suffix(".json.tmp")
    temporary_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_manifest.replace(SMOTER_MANIFEST_PATH)
    smoter_cache_status = "construído"
print("Cache SMOTER:", smoter_cache_status)


## 24. Efeito externo do balanceamento

O SMOTER é comparado ao modelo sem balanceamento nos mesmos eventos OOF. O bootstrap pareado por carro estima `RMSE_SMOTER − RMSE_baseline`; valores negativos favorecem o balanceamento.


In [ ]:
if smoter_oof_predictions.duplicated(["population", "event_id"]).any():
    raise AssertionError("Predições SMOTER duplicadas")
if not smoter_oof_predictions["feature_method"].eq("legacy_fixed_20_smoter").all():
    raise AssertionError("Método inesperado nas predições SMOTER")

smoter_global_rows = []
smoter_bootstrap_rows = []
baseline_fixed = evaluation_oof_predictions[evaluation_oof_predictions["feature_method"].eq("legacy_fixed_20")]
for population_name, smoter_group in smoter_oof_predictions.groupby("population", sort=True):
    smoter_global_rows.append({"population": population_name, "method": "legacy_fixed_20_smoter", "n_events": len(smoter_group), **regression_metrics(smoter_group["soh_real"], smoter_group["soh_pred"])})
    baseline_group = baseline_fixed[baseline_fixed["population"].eq(population_name)][["event_id", "car", "soh_real", "soh_pred"]].rename(columns={"soh_pred": "pred_baseline"})
    paired = baseline_group.merge(smoter_group[["event_id", "soh_pred"]].rename(columns={"soh_pred": "pred_smoter"}), on="event_id", validate="one_to_one")
    cars = paired["car"].drop_duplicates().to_numpy()
    groups = {car: paired.index[paired["car"].eq(car)].to_numpy() for car in cars}
    rng = np.random.default_rng(42); deltas = np.empty(2000)
    for iteration in range(2000):
        sampled = rng.choice(cars, size=len(cars), replace=True)
        indices = np.concatenate([groups[car] for car in sampled])
        y = paired.loc[indices, "soh_real"].to_numpy()
        baseline_rmse = np.sqrt(np.mean((paired.loc[indices, "pred_baseline"].to_numpy()-y)**2))
        smoter_rmse = np.sqrt(np.mean((paired.loc[indices, "pred_smoter"].to_numpy()-y)**2))
        deltas[iteration] = smoter_rmse-baseline_rmse
    observed_baseline = np.sqrt(np.mean((paired["pred_baseline"]-paired["soh_real"])**2))
    observed_smoter = np.sqrt(np.mean((paired["pred_smoter"]-paired["soh_real"])**2))
    smoter_bootstrap_rows.append({
        "population": population_name, "n_events": len(paired), "n_cars": len(cars),
        "rmse_without_smoter": observed_baseline, "rmse_with_smoter": observed_smoter,
        "delta_rmse": observed_smoter-observed_baseline,
        "delta_ci95_low": np.quantile(deltas, .025), "delta_ci95_high": np.quantile(deltas, .975),
        "bootstrap_probability_delta_below_zero": np.mean(deltas < 0),
    })
smoter_global_metrics = pd.DataFrame(smoter_global_rows)
smoter_bootstrap_comparison = pd.DataFrame(smoter_bootstrap_rows)

bin_comparison_rows = []
for population_name, smoter_group in smoter_oof_predictions.groupby("population", sort=True):
    baseline_group = baseline_fixed[baseline_fixed["population"].eq(population_name)][
        ["event_id", "soh_pred"]
    ].rename(columns={"soh_pred": "pred_without_smoter"})
    paired_bins = smoter_group.merge(baseline_group, on="event_id", validate="one_to_one")
    for soh_bin, group in paired_bins.groupby("soh_bin", sort=True):
        rmse_without = np.sqrt(np.mean((group["pred_without_smoter"]-group["soh_real"])**2))
        rmse_with = np.sqrt(np.mean((group["soh_pred"]-group["soh_real"])**2))
        bin_comparison_rows.append({
            "population": population_name, "soh_bin": soh_bin, "n_events": len(group),
            "rmse_without_smoter": rmse_without, "rmse_with_smoter": rmse_with,
            "delta_rmse": rmse_with-rmse_without,
            "smoter_improved": rmse_with < rmse_without,
        })
smoter_bin_comparison = pd.DataFrame(bin_comparison_rows)
smoter_bins_summary = smoter_bin_comparison.groupby("population").agg(
    bins_total=("soh_bin", "size"),
    bins_improved=("smoter_improved", "sum"),
    bins_worsened=("smoter_improved", lambda values: (~values).sum()),
).reset_index()

display(smoter_metrics_by_fold)
display(smoter_global_metrics)
display(smoter_bootstrap_comparison)
display(smoter_balance_plan.groupby("population").agg(
    synthetic_requested=("n_synthetic_requested", "sum"),
    real_train_sum_across_folds=("n_real_train", "sum"),
    eligible_sum_across_bins=("n_eligible_mileage", "sum"),
))
display(smoter_bins_summary)
display(smoter_bin_comparison)


### Resultado da etapa

O impacto do SMOTER foi medido exclusivamente em validações reais, tanto globalmente quanto por faixa de SOH. A decisão de incorporá-lo ao modelo principal dependerá do sinal e do intervalo bootstrap de `ΔRMSE`, com atenção especial aos bins raros.


## 25. Ponderação dos eventos degradados sem SMOTER

Esta análise mantém a configuração `Clean 1 pooled + 20 features fixas + mileage`, sem eventos sintéticos. Os pesos são calculados exclusivamente no treino de cada fold. Para bins degradados (`SOH ≤ 0,955`), usa-se a raiz quadrada da razão entre a frequência do bin mais frequente no treino e a frequência do bin; eventos acima desse limite recebem peso 1. Os pesos são limitados a 3 e normalizados para média 1.

A regra foi definida antes da avaliação e não escolhe pesos a partir dos resultados de validação. A validação permanece sem ponderação e contém somente eventos reais.


In [ ]:
WEIGHTED_STAGE_VERSION = "weighted-v1.0.0-train-bin-invsqrt"
WEIGHTED_DEGRADED_MAX = 0.955
WEIGHTED_MAX_WEIGHT = 3.0
WEIGHTED_METRICS_PATH = CACHE_DIR / "weighted_clean_fixed20_metrics.csv"
WEIGHTED_PREDICTIONS_PATH = CACHE_DIR / "weighted_clean_fixed20_oof_predictions.csv"
WEIGHTED_AUDIT_PATH = CACHE_DIR / "weighted_clean_fixed20_weight_audit.csv"
WEIGHTED_MANIFEST_PATH = CACHE_DIR / "weighted_clean_fixed20.manifest.json"
weighted_inputs = {
    "cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
    "clean": sha256_file(CACHE_DIR / "clean1_pooled_legacy.pkl"),
    "baseline_predictions": sha256_file(EVALUATION_PREDICTIONS_PATH),
}
weighted_parameters = {
    "method": "inverse_sqrt_training_bin_frequency_for_degraded_bins",
    "degraded_max_soh": WEIGHTED_DEGRADED_MAX, "max_weight": WEIGHTED_MAX_WEIGHT,
    "normalization": "training_mean_one", "model": MODEL_PARAMETERS,
}

def training_sample_weights(y_train):
    y_series = pd.Series(np.asarray(y_train, dtype=float))
    bins = pd.cut(y_series, bins=CV_SOH_EDGES, include_lowest=True, right=True)
    counts = bins.value_counts(sort=False)
    reference = float(counts.max())
    mapping = {}
    for interval, count in counts.items():
        degraded = float(interval.right) <= WEIGHTED_DEGRADED_MAX + 1e-12
        raw = np.sqrt(reference / float(count)) if degraded and count > 0 else 1.0
        mapping[interval] = min(float(raw), WEIGHTED_MAX_WEIGHT)
    weights = bins.map(mapping).astype(float).to_numpy()
    weights /= weights.mean()
    return weights, bins, counts, mapping

def weighted_cache_is_compatible():
    required = [WEIGHTED_METRICS_PATH, WEIGHTED_PREDICTIONS_PATH, WEIGHTED_AUDIT_PATH, WEIGHTED_MANIFEST_PATH]
    if not all(path.is_file() for path in required): return False
    manifest = json.loads(WEIGHTED_MANIFEST_PATH.read_text(encoding="utf-8"))
    return (manifest.get("completed") is True and manifest.get("stage_version") == WEIGHTED_STAGE_VERSION
            and manifest.get("input_sha256") == weighted_inputs and manifest.get("parameters") == weighted_parameters
            and manifest.get("predictions_sha256") == sha256_file(WEIGHTED_PREDICTIONS_PATH))

if weighted_cache_is_compatible():
    weighted_metrics_by_fold = pd.read_csv(WEIGHTED_METRICS_PATH)
    weighted_oof_predictions = pd.read_csv(WEIGHTED_PREDICTIONS_PATH)
    weighted_weight_audit = pd.read_csv(WEIGHTED_AUDIT_PATH)
    weighted_cache_status = "reutilizado"
else:
    metric_rows, prediction_frames, audit_rows = [], [], []
    model_features = LEGACY_FEATURES_20 + ["mileage"]
    start_total = time.perf_counter()
    for fold_id in range(CV_N_SPLITS):
        train = cv_clean1_pooled_legacy.loc[~cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        validation = cv_clean1_pooled_legacy.loc[cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        X_train = train[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        X_validation = validation[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        medians = X_train.median(axis=0); X_train = X_train.fillna(medians); X_validation = X_validation.fillna(medians)
        y_train = train[TARGET_COL].to_numpy(dtype=float); y_validation = validation[TARGET_COL].to_numpy(dtype=float)
        sample_weight, train_bins, counts, mapping = training_sample_weights(y_train)
        start = time.perf_counter(); model = ExtraTreesRegressor(**MODEL_PARAMETERS)
        model.fit(X_train, y_train, sample_weight=sample_weight); predictions = model.predict(X_validation)
        metrics = regression_metrics(y_validation, predictions)
        metric_rows.append({"population": "clean1_pooled", "feature_method": "legacy_fixed_20_weighted",
            "fold": fold_id, "n_train": len(train), "n_validation": len(validation),
            "weight_min": sample_weight.min(), "weight_max": sample_weight.max(),
            "fit_predict_seconds": time.perf_counter()-start, **metrics})
        prediction_frames.append(pd.DataFrame({"population": "clean1_pooled",
            "feature_method": "legacy_fixed_20_weighted", "fold": fold_id,
            "event_id": validation["event_id"].to_numpy(), "car": validation["car"].to_numpy(),
            "mileage": validation["mileage"].to_numpy(), "soh_real": y_validation,
            "soh_pred": predictions, "error": predictions-y_validation,
            "abs_error": np.abs(predictions-y_validation)}))
        for interval, count in counts.items():
            audit_rows.append({"fold": fold_id, "soh_bin": str(interval), "n_train": int(count),
                "raw_weight": float(mapping[interval]),
                "normalized_weight": float(mapping[interval] / np.mean([mapping[item] for item in train_bins]))})
        print(f"fold {fold_id}: RMSE={metrics['rmse']:.6f}; peso máximo={sample_weight.max():.3f}")
        del model, X_train, X_validation, train, validation
    weighted_metrics_by_fold = pd.DataFrame(metric_rows); weighted_oof_predictions = pd.concat(prediction_frames, ignore_index=True)
    weighted_weight_audit = pd.DataFrame(audit_rows)
    for frame, path in [(weighted_metrics_by_fold, WEIGHTED_METRICS_PATH), (weighted_oof_predictions, WEIGHTED_PREDICTIONS_PATH), (weighted_weight_audit, WEIGHTED_AUDIT_PATH)]:
        temporary = path.with_suffix(path.suffix + ".tmp"); frame.to_csv(temporary, index=False); temporary.replace(path)
    manifest = {"stage": "weighted_clean_fixed20", "stage_version": WEIGHTED_STAGE_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "completed": True,
        "parameters": weighted_parameters, "input_sha256": weighted_inputs,
        "metrics_shape": list(weighted_metrics_by_fold.shape), "predictions_shape": list(weighted_oof_predictions.shape),
        "metrics_sha256": sha256_file(WEIGHTED_METRICS_PATH),
        "predictions_sha256": sha256_file(WEIGHTED_PREDICTIONS_PATH),
        "weight_audit_sha256": sha256_file(WEIGHTED_AUDIT_PATH), "elapsed_seconds": time.perf_counter()-start_total}
    temporary = WEIGHTED_MANIFEST_PATH.with_suffix(".json.tmp"); temporary.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"); temporary.replace(WEIGHTED_MANIFEST_PATH)
    weighted_cache_status = "construído"
print("Cache ponderado:", weighted_cache_status)


In [ ]:
baseline_clean = evaluation_oof_predictions[(evaluation_oof_predictions["population"].eq("clean1_pooled")) & (evaluation_oof_predictions["feature_method"].eq("legacy_fixed_20"))].copy()
paired_weighted = baseline_clean[["event_id", "car", "soh_real", "soh_pred"]].rename(columns={"soh_pred": "pred_baseline"}).merge(
    weighted_oof_predictions[["event_id", "soh_pred"]].rename(columns={"soh_pred": "pred_weighted"}), on="event_id", validate="one_to_one")
weighted_global_comparison = pd.DataFrame([
    {"method": "baseline_no_smoter", **regression_metrics(paired_weighted["soh_real"], paired_weighted["pred_baseline"])},
    {"method": "weighted_no_smoter", **regression_metrics(paired_weighted["soh_real"], paired_weighted["pred_weighted"])},
])
paired_weighted["soh_bin"] = pd.cut(paired_weighted["soh_real"], bins=CV_SOH_EDGES, include_lowest=True, right=True)
bin_rows = []
for interval, group in paired_weighted.groupby("soh_bin", observed=False, sort=True):
    if len(group) == 0: continue
    base = regression_metrics(group["soh_real"], group["pred_baseline"]); weighted = regression_metrics(group["soh_real"], group["pred_weighted"])
    bin_rows.append({"soh_bin": str(interval), "n_events": len(group), "rmse_baseline": base["rmse"],
        "rmse_weighted": weighted["rmse"], "delta_rmse": weighted["rmse"]-base["rmse"],
        "mae_baseline": base["mae"], "mae_weighted": weighted["mae"],
        "bias_baseline": base["bias"], "bias_weighted": weighted["bias"]})
weighted_metrics_by_bin = pd.DataFrame(bin_rows)
cars = paired_weighted["car"].drop_duplicates().to_numpy(); groups = {car: paired_weighted.index[paired_weighted["car"].eq(car)].to_numpy() for car in cars}
rng = np.random.default_rng(42); deltas = np.empty(2000)
for iteration in range(2000):
    sampled = rng.choice(cars, size=len(cars), replace=True); indices = np.concatenate([groups[car] for car in sampled]); y = paired_weighted.loc[indices, "soh_real"].to_numpy()
    deltas[iteration] = np.sqrt(np.mean((paired_weighted.loc[indices, "pred_weighted"].to_numpy()-y)**2)) - np.sqrt(np.mean((paired_weighted.loc[indices, "pred_baseline"].to_numpy()-y)**2))
weighted_bootstrap = pd.DataFrame([{"comparison": "weighted_minus_baseline", "delta_rmse": weighted_global_comparison.loc[1, "rmse"]-weighted_global_comparison.loc[0, "rmse"],
    "ci95_low": np.quantile(deltas, 0.025), "ci95_high": np.quantile(deltas, 0.975), "p_delta_below_zero": np.mean(deltas < 0)}])
display(weighted_metrics_by_fold); display(weighted_global_comparison); display(weighted_bootstrap); display(weighted_metrics_by_bin); display(weighted_weight_audit)


### Resultado da ponderação

A ponderação inversa pela raiz da frequência **não foi favorável**. O RMSE OOF global aumentou de `0,007983` para `0,008076` (`Δ = +0,000093`; bootstrap pareado por carro IC95% `[+0,000072; +0,000112]`). Nos 1.220 eventos degradados com `SOH ≤ 0,955`, o RMSE aumentou de `0,013854` para `0,014003`. Somente o bin `(0,925; 0,930]` apresentou pequena melhora de RMSE; os demais bins degradados pioraram.

Portanto, esta configuração ponderada não substituirá o baseline `Clean 1 pooled + 20 features fixas + mileage`, sem SMOTER. Testar outros pesos usando estes mesmos folds externos para escolher o melhor valor configuraria ajuste sobre a validação; qualquer otimização adicional deve ser aninhada dentro de cada treino.


## 26. Ensemble global–especialista para degradação

O evento degradado de interesse é definido previamente como `SOH ≤ 0,94`. Como o SOH verdadeiro não existe na inferência, ele nunca é usado para escolher o modelo na validação. Em cada fold externo, um gate aprende no treino a probabilidade de o evento pertencer à região baixa ou de transição (`SOH ≤ 0,95`).

O especialista é treinado com eventos reais até `SOH ≤ 0,95`, fornecendo margem em torno da fronteira de interesse. São avaliadas duas versões: especialista sem SMOTER e especialista com SMOTER condicionado. A combinação é suave: `ŷ = (1-p)ŷ_global + pŷ_especialista`, em que `p` é produzido pelo gate usando somente as features disponíveis.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

EXPERT_STAGE_VERSION = "experts-v1.0.0-soft-gate-094-095"
DEGRADED_SOH_MAX = 0.94
EXPERT_SUPPORT_MAX = 0.95
GATE_PARAMETERS = {
    "n_estimators": 800, "max_depth": None, "min_samples_split": 2,
    "min_samples_leaf": 2, "max_features": 1.0, "bootstrap": False,
    "class_weight": "balanced", "random_state": 42, "n_jobs": -1,
}
EXPERT_PREDICTIONS_PATH = CACHE_DIR / "experts_clean_fixed20_oof_predictions.csv"
EXPERT_FOLD_METRICS_PATH = CACHE_DIR / "experts_clean_fixed20_fold_metrics.csv"
EXPERT_SMOTER_AUDIT_PATH = CACHE_DIR / "experts_smoter_audit.csv"
EXPERT_MANIFEST_PATH = CACHE_DIR / "experts_clean_fixed20.manifest.json"
expert_inputs = {
    "cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
    "clean": sha256_file(CACHE_DIR / "clean1_pooled_legacy.pkl"),
    "baseline_predictions": sha256_file(EVALUATION_PREDICTIONS_PATH),
}
expert_parameters = {"degraded_soh_max": DEGRADED_SOH_MAX, "expert_support_max": EXPERT_SUPPORT_MAX,
    "combination": "soft_probability", "model": MODEL_PARAMETERS, "gate": GATE_PARAMETERS,
    "smoter_stage_version": SMOTER_STAGE_VERSION}

def expert_cache_is_compatible():
    required = [EXPERT_PREDICTIONS_PATH, EXPERT_FOLD_METRICS_PATH, EXPERT_SMOTER_AUDIT_PATH, EXPERT_MANIFEST_PATH]
    if not all(path.is_file() for path in required): return False
    manifest = json.loads(EXPERT_MANIFEST_PATH.read_text(encoding="utf-8"))
    return (manifest.get("completed") is True and manifest.get("stage_version") == EXPERT_STAGE_VERSION
            and manifest.get("input_sha256") == expert_inputs and manifest.get("parameters") == expert_parameters
            and manifest.get("predictions_sha256") == sha256_file(EXPERT_PREDICTIONS_PATH))

if expert_cache_is_compatible():
    expert_oof_predictions = pd.read_csv(EXPERT_PREDICTIONS_PATH)
    expert_fold_metrics = pd.read_csv(EXPERT_FOLD_METRICS_PATH)
    expert_smoter_audit = pd.read_csv(EXPERT_SMOTER_AUDIT_PATH)
    expert_cache_status = "reutilizado"
else:
    prediction_frames, fold_rows, smoter_audit_frames = [], [], []
    model_features = LEGACY_FEATURES_20 + ["mileage"]
    baseline_clean = evaluation_oof_predictions[(evaluation_oof_predictions["population"].eq("clean1_pooled")) & (evaluation_oof_predictions["feature_method"].eq("legacy_fixed_20"))]
    total_start = time.perf_counter()
    for fold_id in range(CV_N_SPLITS):
        train = cv_clean1_pooled_legacy.loc[~cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        validation = cv_clean1_pooled_legacy.loc[cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        X_train = train[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        X_validation = validation[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        medians = X_train.median(axis=0); X_train = X_train.fillna(medians); X_validation = X_validation.fillna(medians)
        gate_y = train[TARGET_COL].le(EXPERT_SUPPORT_MAX).astype(int).to_numpy()
        gate = ExtraTreesClassifier(**GATE_PARAMETERS); gate.fit(X_train, gate_y)
        gate_probability = gate.predict_proba(X_validation)[:, list(gate.classes_).index(1)]
        expert_real = train.loc[train[TARGET_COL].le(EXPERT_SUPPORT_MAX)].copy()
        X_expert = expert_real[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(medians)
        expert_plain_model = ExtraTreesRegressor(**MODEL_PARAMETERS); expert_plain_model.fit(X_expert, expert_real[TARGET_COL].to_numpy(dtype=float))
        pred_expert_plain = expert_plain_model.predict(X_validation)
        expert_real["soh_bin"] = pd.cut(expert_real[TARGET_COL], bins=CV_SOH_EDGES, include_lowest=True, right=True)
        expert_smoter, balance, generation, neighbors = build_smoter_training(expert_real, "clean1_expert", fold_id, legacy_candidate_features)
        X_expert_smoter = expert_smoter[model_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(medians)
        expert_smoter_model = ExtraTreesRegressor(**MODEL_PARAMETERS); expert_smoter_model.fit(X_expert_smoter, expert_smoter[TARGET_COL].to_numpy(dtype=float))
        pred_expert_smoter = expert_smoter_model.predict(X_validation)
        base_fold = baseline_clean.loc[baseline_clean["fold"].eq(fold_id), ["event_id", "soh_pred"]].rename(columns={"soh_pred": "pred_global"})
        aligned = validation[["event_id", "car", "mileage", TARGET_COL]].merge(base_fold, on="event_id", validate="one_to_one", sort=False)
        pred_plain = (1.0-gate_probability)*aligned["pred_global"].to_numpy() + gate_probability*pred_expert_plain
        pred_smoter = (1.0-gate_probability)*aligned["pred_global"].to_numpy() + gate_probability*pred_expert_smoter
        fold_prediction = aligned.rename(columns={TARGET_COL: "soh_real"}); fold_prediction["fold"] = fold_id
        fold_prediction["gate_probability"] = gate_probability; fold_prediction["pred_expert_plain"] = pred_expert_plain
        fold_prediction["pred_expert_smoter"] = pred_expert_smoter; fold_prediction["pred_ensemble_plain"] = pred_plain
        fold_prediction["pred_ensemble_smoter"] = pred_smoter; prediction_frames.append(fold_prediction)
        degraded_mask = fold_prediction["soh_real"].le(DEGRADED_SOH_MAX)
        for method, pred in [("global", fold_prediction["pred_global"]), ("ensemble_plain", pred_plain), ("ensemble_smoter", pred_smoter)]:
            fold_rows.append({"fold": fold_id, "method": method, "n_validation": len(validation),
                "n_degraded": int(degraded_mask.sum()), **regression_metrics(fold_prediction["soh_real"], pred),
                "degraded_rmse": regression_metrics(fold_prediction.loc[degraded_mask, "soh_real"], np.asarray(pred)[degraded_mask.to_numpy()])["rmse"]})
        balance["audit_type"] = "balance"; smoter_audit_frames.append(balance)
        print(f"fold {fold_id}: treino especialista={len(expert_real)}, sintéticos={len(expert_smoter)-len(expert_real)}, degradados validação={degraded_mask.sum()}")
        del gate, expert_plain_model, expert_smoter_model, X_train, X_validation, X_expert, X_expert_smoter
    expert_oof_predictions = pd.concat(prediction_frames, ignore_index=True); expert_fold_metrics = pd.DataFrame(fold_rows)
    expert_smoter_audit = pd.concat(smoter_audit_frames, ignore_index=True)
    if expert_oof_predictions["event_id"].duplicated().any(): raise AssertionError("Predições OOF duplicadas")
    for frame, path in [(expert_oof_predictions, EXPERT_PREDICTIONS_PATH), (expert_fold_metrics, EXPERT_FOLD_METRICS_PATH), (expert_smoter_audit, EXPERT_SMOTER_AUDIT_PATH)]:
        temporary=path.with_suffix(path.suffix+".tmp"); frame.to_csv(temporary,index=False); temporary.replace(path)
    manifest={"stage":"global_specialist_ensemble","stage_version":EXPERT_STAGE_VERSION,"created_at_utc":datetime.now(timezone.utc).isoformat(),"completed":True,
        "parameters":expert_parameters,"input_sha256":expert_inputs,"predictions_shape":list(expert_oof_predictions.shape),
        "predictions_sha256":sha256_file(EXPERT_PREDICTIONS_PATH),"fold_metrics_sha256":sha256_file(EXPERT_FOLD_METRICS_PATH),
        "smoter_audit_sha256":sha256_file(EXPERT_SMOTER_AUDIT_PATH),"elapsed_seconds":time.perf_counter()-total_start}
    temporary=EXPERT_MANIFEST_PATH.with_suffix(".json.tmp"); temporary.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8"); temporary.replace(EXPERT_MANIFEST_PATH)
    expert_cache_status="construído"
print("Cache ensemble especialista:",expert_cache_status)


In [ ]:
expert_global_rows=[]
for method,column in [("global","pred_global"),("ensemble_plain","pred_ensemble_plain"),("ensemble_smoter","pred_ensemble_smoter")]:
    all_metrics=regression_metrics(expert_oof_predictions["soh_real"],expert_oof_predictions[column])
    low=expert_oof_predictions[expert_oof_predictions["soh_real"].le(DEGRADED_SOH_MAX)]
    low_metrics=regression_metrics(low["soh_real"],low[column])
    expert_global_rows.append({"method":method,"n_events":len(expert_oof_predictions),"n_degraded":len(low),
        **all_metrics,"degraded_rmse":low_metrics["rmse"],"degraded_mae":low_metrics["mae"],"degraded_bias":low_metrics["bias"]})
expert_global_metrics=pd.DataFrame(expert_global_rows)
gate_truth=expert_oof_predictions["soh_real"].le(EXPERT_SUPPORT_MAX).astype(int); gate_pred=expert_oof_predictions["gate_probability"].ge(0.5).astype(int)
gate_metrics=pd.DataFrame([{"threshold":0.5,"positive_definition":f"SOH <= {EXPERT_SUPPORT_MAX}",
    "precision":precision_score(gate_truth,gate_pred,zero_division=0),"recall":recall_score(gate_truth,gate_pred,zero_division=0),
    "f1":f1_score(gate_truth,gate_pred,zero_division=0),"roc_auc":roc_auc_score(gate_truth,expert_oof_predictions["gate_probability"]),
    "average_precision":average_precision_score(gate_truth,expert_oof_predictions["gate_probability"]) }])
bin_rows=[]; expert_oof_predictions["soh_bin"]=pd.cut(expert_oof_predictions["soh_real"],bins=CV_SOH_EDGES,include_lowest=True,right=True)
for interval,group in expert_oof_predictions.groupby("soh_bin",observed=False,sort=True):
    if not len(group): continue
    row={"soh_bin":str(interval),"n_events":len(group),"mean_gate_probability":group["gate_probability"].mean()}
    for method,column in [("global","pred_global"),("plain","pred_ensemble_plain"),("smoter","pred_ensemble_smoter")]: row[f"rmse_{method}"]=regression_metrics(group["soh_real"],group[column])["rmse"]
    bin_rows.append(row)
expert_metrics_by_bin=pd.DataFrame(bin_rows)
display(expert_fold_metrics);display(expert_global_metrics);display(gate_metrics);display(expert_metrics_by_bin);display(expert_smoter_audit.groupby("fold").agg(real_low_train=("n_real_train","sum"),synthetic=("n_synthetic_requested","sum")))


### Resultado do ensemble especialista

O ensemble reduziu substancialmente o erro na região de interesse, com custo no desempenho global. Em `SOH ≤ 0,94` (403 eventos), o RMSE passou de `0,016655` no modelo global para `0,012685` no ensemble sem SMOTER e `0,012352` no ensemble com SMOTER. Globalmente, os respectivos RMSE foram `0,007983`, `0,008237` e `0,008272`.

O gate obteve ROC AUC `0,9704`, average precision `0,8027`, precisão `0,7805` e recall `0,7128` para a região `SOH ≤ 0,95`. Assim, o ensemble com SMOTER oferece o melhor desempenho para detectar a degradação definida pelo estudo, mas deve ser apresentado como modelo orientado à região degradada, acompanhado explicitamente de sua perda global.


## 27. Ensemble residual por Δ-learning

O especialista deixa de estimar diretamente o SOH e aprende somente o erro residual do modelo global. Para impedir resíduos artificialmente pequenos, as predições globais usadas como target residual são geradas por cinco folds internos dentro de cada treino externo. Nenhum evento da validação externa participa do gate, do cálculo dos resíduos, do especialista ou do SMOTER.

A predição final é `ŷ = ŷ_global + p_gate·Δ̂`. O gate reconhece a região de suporte `SOH ≤ 0,95`, enquanto o desfecho degradado principal permanece `SOH ≤ 0,94`. São comparados especialistas residuais com e sem SMOTER.


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=r"`sklearn\.utils\.parallel\.delayed` should be used.*", category=UserWarning)
DELTA_STAGE_VERSION = "delta-v1.0.0-nested-crossfit-soft-gate"
DELTA_INNER_SPLITS = 5
DELTA_INNER_RANDOM_STATE = 2026
DELTA_PREDICTIONS_PATH = CACHE_DIR / "delta_experts_clean_fixed20_oof_predictions.csv"
DELTA_FOLD_METRICS_PATH = CACHE_DIR / "delta_experts_clean_fixed20_fold_metrics.csv"
DELTA_SMOTER_AUDIT_PATH = CACHE_DIR / "delta_experts_smoter_audit.csv"
DELTA_MANIFEST_PATH = CACHE_DIR / "delta_experts_clean_fixed20.manifest.json"
delta_inputs = {"cv_assignments": sha256_file(CV_ASSIGNMENTS_PATH),
    "clean": sha256_file(CACHE_DIR / "clean1_pooled_legacy.pkl"),
    "baseline_predictions": sha256_file(EVALUATION_PREDICTIONS_PATH)}
delta_parameters = {"degraded_soh_max": DEGRADED_SOH_MAX, "expert_support_max": EXPERT_SUPPORT_MAX,
    "inner_splits": DELTA_INNER_SPLITS, "inner_random_state": DELTA_INNER_RANDOM_STATE,
    "residual_source": "inner_cross_fitted_global", "combination": "global_plus_gate_probability_times_delta",
    "model": MODEL_PARAMETERS, "gate": GATE_PARAMETERS, "smoter_stage_version": SMOTER_STAGE_VERSION}

def delta_cache_is_compatible():
    required=[DELTA_PREDICTIONS_PATH,DELTA_FOLD_METRICS_PATH,DELTA_SMOTER_AUDIT_PATH,DELTA_MANIFEST_PATH]
    if not all(path.is_file() for path in required): return False
    manifest=json.loads(DELTA_MANIFEST_PATH.read_text(encoding="utf-8"))
    return (manifest.get("completed") is True and manifest.get("stage_version")==DELTA_STAGE_VERSION
        and manifest.get("input_sha256")==delta_inputs and manifest.get("parameters")==delta_parameters
        and manifest.get("predictions_sha256")==sha256_file(DELTA_PREDICTIONS_PATH))

def inner_cross_fitted_global_predictions(outer_train, model_features, outer_fold):
    labels=pd.cut(outer_train[TARGET_COL],bins=CV_SOH_EDGES,include_lowest=True,right=True).astype(str).to_numpy()
    splitter=StratifiedKFold(n_splits=DELTA_INNER_SPLITS,shuffle=True,random_state=DELTA_INNER_RANDOM_STATE+outer_fold)
    predictions=pd.Series(np.nan,index=outer_train.index,dtype=float)
    for inner_fold,(fit_positions,hold_positions) in enumerate(splitter.split(np.zeros(len(outer_train),dtype=np.uint8),labels)):
        fit=outer_train.iloc[fit_positions]; hold=outer_train.iloc[hold_positions]
        X_fit=fit[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
        X_hold=hold[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
        medians=X_fit.median(axis=0)
        if medians.isna().any(): raise ValueError(f"Medianas internas ausentes: fold {outer_fold}/{inner_fold}")
        model=ExtraTreesRegressor(**MODEL_PARAMETERS); model.fit(X_fit.fillna(medians),fit[TARGET_COL].to_numpy(dtype=float))
        predictions.loc[hold.index]=model.predict(X_hold.fillna(medians)); del model,X_fit,X_hold
    if predictions.isna().any(): raise AssertionError(f"Cross-fitting incompleto no fold externo {outer_fold}")
    return predictions

if delta_cache_is_compatible():
    delta_oof_predictions=pd.read_csv(DELTA_PREDICTIONS_PATH); delta_fold_metrics=pd.read_csv(DELTA_FOLD_METRICS_PATH)
    delta_smoter_audit=pd.read_csv(DELTA_SMOTER_AUDIT_PATH); delta_cache_status="reutilizado"
else:
    prediction_frames,fold_rows,smoter_audits=[],[],[]; model_features=LEGACY_FEATURES_20+["mileage"]
    baseline_clean=evaluation_oof_predictions[(evaluation_oof_predictions["population"].eq("clean1_pooled")) & (evaluation_oof_predictions["feature_method"].eq("legacy_fixed_20"))]
    total_start=time.perf_counter()
    for fold_id in range(CV_N_SPLITS):
        fold_start=time.perf_counter(); train=cv_clean1_pooled_legacy.loc[~cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        validation=cv_clean1_pooled_legacy.loc[cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        crossfit_pred=inner_cross_fitted_global_predictions(train,model_features,fold_id)
        train["global_crossfit_pred"]=crossfit_pred; train["delta_target"]=train[TARGET_COL]-train["global_crossfit_pred"]
        X_train=train[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
        X_validation=validation[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan)
        medians=X_train.median(axis=0); X_train=X_train.fillna(medians); X_validation=X_validation.fillna(medians)
        gate=ExtraTreesClassifier(**GATE_PARAMETERS); gate.fit(X_train,train[TARGET_COL].le(EXPERT_SUPPORT_MAX).astype(int))
        gate_probability=gate.predict_proba(X_validation)[:,list(gate.classes_).index(1)]
        residual_train=train.loc[train[TARGET_COL].le(EXPERT_SUPPORT_MAX)].copy()
        X_residual=residual_train[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(medians)
        plain_model=ExtraTreesRegressor(**MODEL_PARAMETERS); plain_model.fit(X_residual,residual_train["delta_target"].to_numpy(dtype=float))
        delta_plain=plain_model.predict(X_validation)
        residual_train["soh_original_for_audit"]=residual_train[TARGET_COL]
        residual_train["soh_bin"]=pd.cut(residual_train[TARGET_COL],bins=CV_SOH_EDGES,include_lowest=True,right=True)
        residual_train[TARGET_COL]=residual_train["delta_target"]
        residual_smoter,balance,generation,neighbors=build_smoter_training(residual_train,"clean1_delta_expert",fold_id,legacy_candidate_features)
        X_residual_smoter=residual_smoter[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(medians)
        smoter_model=ExtraTreesRegressor(**MODEL_PARAMETERS); smoter_model.fit(X_residual_smoter,residual_smoter[TARGET_COL].to_numpy(dtype=float))
        delta_smoter=smoter_model.predict(X_validation)
        base_fold=baseline_clean.loc[baseline_clean["fold"].eq(fold_id),["event_id","soh_pred"]].rename(columns={"soh_pred":"pred_global"})
        aligned=validation[["event_id","car","mileage",TARGET_COL]].merge(base_fold,on="event_id",validate="one_to_one",sort=False)
        pred_plain=aligned["pred_global"].to_numpy()+gate_probability*delta_plain
        pred_smoter=aligned["pred_global"].to_numpy()+gate_probability*delta_smoter
        result=aligned.rename(columns={TARGET_COL:"soh_real"}); result["fold"]=fold_id; result["gate_probability"]=gate_probability
        result["delta_plain"]=delta_plain; result["delta_smoter"]=delta_smoter
        result["pred_delta_plain"]=pred_plain; result["pred_delta_smoter"]=pred_smoter; prediction_frames.append(result)
        degraded=result["soh_real"].le(DEGRADED_SOH_MAX)
        for method,pred in [("global",result["pred_global"]),("delta_plain",pred_plain),("delta_smoter",pred_smoter)]:
            low_metrics=regression_metrics(result.loc[degraded,"soh_real"],np.asarray(pred)[degraded.to_numpy()])
            fold_rows.append({"fold":fold_id,"method":method,"n_validation":len(result),"n_degraded":int(degraded.sum()),
                **regression_metrics(result["soh_real"],pred),"degraded_rmse":low_metrics["rmse"],"degraded_mae":low_metrics["mae"],"degraded_bias":low_metrics["bias"]})
        balance["crossfit_residual_rmse"]=float(np.sqrt(np.mean(train["delta_target"]**2))); smoter_audits.append(balance)
        print(f"fold {fold_id}: residual real={len(residual_train)}, sintéticos={len(residual_smoter)-len(residual_train)}, tempo={time.perf_counter()-fold_start:.1f}s")
        del gate,plain_model,smoter_model,X_train,X_validation,X_residual,X_residual_smoter
    delta_oof_predictions=pd.concat(prediction_frames,ignore_index=True); delta_fold_metrics=pd.DataFrame(fold_rows); delta_smoter_audit=pd.concat(smoter_audits,ignore_index=True)
    if delta_oof_predictions["event_id"].duplicated().any(): raise AssertionError("Predições delta OOF duplicadas")
    for frame,path in [(delta_oof_predictions,DELTA_PREDICTIONS_PATH),(delta_fold_metrics,DELTA_FOLD_METRICS_PATH),(delta_smoter_audit,DELTA_SMOTER_AUDIT_PATH)]:
        temporary=path.with_suffix(path.suffix+".tmp"); frame.to_csv(temporary,index=False); temporary.replace(path)
    manifest={"stage":"nested_residual_delta_ensemble","stage_version":DELTA_STAGE_VERSION,"created_at_utc":datetime.now(timezone.utc).isoformat(),"completed":True,
        "parameters":delta_parameters,"input_sha256":delta_inputs,"predictions_shape":list(delta_oof_predictions.shape),
        "predictions_sha256":sha256_file(DELTA_PREDICTIONS_PATH),"fold_metrics_sha256":sha256_file(DELTA_FOLD_METRICS_PATH),
        "smoter_audit_sha256":sha256_file(DELTA_SMOTER_AUDIT_PATH),"elapsed_seconds":time.perf_counter()-total_start}
    temporary=DELTA_MANIFEST_PATH.with_suffix(".json.tmp"); temporary.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8"); temporary.replace(DELTA_MANIFEST_PATH); delta_cache_status="construído"
print("Cache Δ-learning:",delta_cache_status)


In [ ]:
delta_global_rows=[]
for method,column in [("global","pred_global"),("delta_plain","pred_delta_plain"),("delta_smoter","pred_delta_smoter")]:
    low=delta_oof_predictions[delta_oof_predictions["soh_real"].le(DEGRADED_SOH_MAX)]
    all_metrics=regression_metrics(delta_oof_predictions["soh_real"],delta_oof_predictions[column]); low_metrics=regression_metrics(low["soh_real"],low[column])
    delta_global_rows.append({"method":method,"n_events":len(delta_oof_predictions),"n_degraded":len(low),**all_metrics,
        "degraded_rmse":low_metrics["rmse"],"degraded_mae":low_metrics["mae"],"degraded_bias":low_metrics["bias"]})
delta_global_metrics=pd.DataFrame(delta_global_rows)
delta_oof_predictions["soh_bin"]=pd.cut(delta_oof_predictions["soh_real"],bins=CV_SOH_EDGES,include_lowest=True,right=True)
delta_bin_rows=[]
for interval,group in delta_oof_predictions.groupby("soh_bin",observed=False,sort=True):
    if not len(group): continue
    row={"soh_bin":str(interval),"n_events":len(group),"gate_probability":group["gate_probability"].mean()}
    for method,column in [("global","pred_global"),("delta_plain","pred_delta_plain"),("delta_smoter","pred_delta_smoter")]: row[f"rmse_{method}"]=regression_metrics(group["soh_real"],group[column])["rmse"]
    delta_bin_rows.append(row)
delta_metrics_by_bin=pd.DataFrame(delta_bin_rows)
display(delta_fold_metrics);display(delta_global_metrics);display(delta_metrics_by_bin);display(delta_smoter_audit.groupby("fold").agg(real_residual_train=("n_real_train","sum"),synthetic=("n_synthetic_requested","sum"),crossfit_residual_rmse=("crossfit_residual_rmse","first")))


### Resultado do Δ-learning

O Δ-learning apresentou uma relação mais favorável entre desempenho global e erro degradado. O baseline global obteve RMSE `0,007983` no conjunto completo e `0,016655` nos 403 eventos com `SOH ≤ 0,94`. O especialista residual sem SMOTER obteve `0,008052` e `0,012654`, respectivamente; com SMOTER residual, `0,008078` e `0,012575`.

Em relação ao especialista direto, o residual sem SMOTER preservou praticamente o ganho na região degradada (`0,012654` contra `0,012685`) e reduziu substancialmente a penalização global (`0,008052` contra `0,008237`). O SMOTER residual produziu ganho degradado adicional pequeno (`0,000079` de RMSE) acompanhado de piora global de `0,000026`; essa diferença deve ser tratada como secundária até a comparação pareada por carro.


## 28. Incertezas U1, U2 e U3: global e ensemble residual

A avaliação compara o modelo global e o ensemble residual sem SMOTER. Em cada fold externo, 20% do treino é reservado por estratificação para calibração; os 80% restantes formam o núcleo de ajuste. O especialista aprende resíduos globais cross-fitted exclusivamente nesse núcleo. A validação externa não participa de nenhuma decisão.

- **U1:** distribuição Monte Carlo das árvores. No ensemble, são propagadas conjuntamente árvores do global, do gate e do especialista residual. É uma medida diagnóstica, sem garantia formal de cobertura.
- **U2:** conformal simétrico com scores absolutos do ensemble final na calibração.
- **U3:** conformal adaptativo, normalizando o erro pela escala U1 específica do evento.

Os intervalos não são truncados à faixa observada de SOH, pois truncamento posterior poderia alterar a cobertura. Resultados condicionados ao SOH verdadeiro são usados apenas para auditoria, nunca para construir os intervalos.


In [ ]:
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore", message=r"X has feature names, but ExtraTree.* was fitted without feature names", category=UserWarning)

UNCERTAINTY_STAGE_VERSION = "uncertainty-v1.2.0-u3-mondrian-gate-90-95"
UNCERTAINTY_ALPHA = 0.05
UNCERTAINTY_ALPHA_90 = 0.10
UNCERTAINTY_CALIBRATION_FRACTION = 0.20
UNCERTAINTY_RANDOM_STATE = 31415
UNCERTAINTY_MC_DRAWS = 400
UNCERTAINTY_SCALE_FLOOR = 1e-4
MONDRIAN_GATE_EDGES = [-np.inf, 0.25, 0.75, np.inf]
MONDRIAN_GATE_LABELS = ["common", "transition", "likely_degraded"]
MONDRIAN_MIN_CALIBRATION = 50
UNCERTAINTY_PREDICTIONS_PATH = CACHE_DIR / "uncertainty_global_delta_oof_predictions.csv"
UNCERTAINTY_FOLD_PATH = CACHE_DIR / "uncertainty_global_delta_fold_audit.csv"
UNCERTAINTY_MANIFEST_PATH = CACHE_DIR / "uncertainty_global_delta.manifest.json"
uncertainty_inputs = {"cv_assignments":sha256_file(CV_ASSIGNMENTS_PATH),"clean":sha256_file(CACHE_DIR/"clean1_pooled_legacy.pkl")}
uncertainty_parameters = {"nominal_levels":[0.90,0.95],"alpha_95":UNCERTAINTY_ALPHA,"alpha_90":UNCERTAINTY_ALPHA_90,"calibration_fraction":UNCERTAINTY_CALIBRATION_FRACTION,
    "random_state":UNCERTAINTY_RANDOM_STATE,"mc_draws":UNCERTAINTY_MC_DRAWS,"scale_floor":UNCERTAINTY_SCALE_FLOOR,
    "model":MODEL_PARAMETERS,"gate":GATE_PARAMETERS,"expert_support_max":EXPERT_SUPPORT_MAX,"degraded_soh_max":DEGRADED_SOH_MAX,
    "mondrian_gate_edges":[0.25,0.75],"mondrian_labels":MONDRIAN_GATE_LABELS,"mondrian_min_calibration":MONDRIAN_MIN_CALIBRATION}

def conformal_quantile(scores,alpha=UNCERTAINTY_ALPHA):
    values=np.sort(np.asarray(scores,dtype=float)); rank=int(np.ceil((len(values)+1)*(1-alpha)))
    return float(values[min(max(rank,1),len(values))-1])

def regression_tree_draws(model,X,draw_indices):
    return np.vstack([model.estimators_[int(index)].predict(X) for index in draw_indices])

def gate_tree_draws(model,X,draw_indices):
    rows=[]
    for index in draw_indices:
        tree=model.estimators_[int(index)]; probabilities=tree.predict_proba(X)
        classes=list(tree.classes_); rows.append(probabilities[:,classes.index(1)] if 1 in classes else np.zeros(len(X)))
    return np.vstack(rows)

def uncertainty_cache_is_compatible():
    required=[UNCERTAINTY_PREDICTIONS_PATH,UNCERTAINTY_FOLD_PATH,UNCERTAINTY_MANIFEST_PATH]
    if not all(path.is_file() for path in required): return False
    manifest=json.loads(UNCERTAINTY_MANIFEST_PATH.read_text(encoding="utf-8"))
    return (manifest.get("completed") is True and manifest.get("stage_version")==UNCERTAINTY_STAGE_VERSION
        and manifest.get("input_sha256")==uncertainty_inputs and manifest.get("parameters")==uncertainty_parameters
        and manifest.get("predictions_sha256")==sha256_file(UNCERTAINTY_PREDICTIONS_PATH))

if uncertainty_cache_is_compatible():
    uncertainty_oof_predictions=pd.read_csv(UNCERTAINTY_PREDICTIONS_PATH); uncertainty_fold_audit=pd.read_csv(UNCERTAINTY_FOLD_PATH); uncertainty_cache_status="reutilizado"
else:
    prediction_frames,fold_audits=[],[]; model_features=LEGACY_FEATURES_20+["mileage"]; total_start=time.perf_counter()
    for fold_id in range(CV_N_SPLITS):
        fold_start=time.perf_counter(); outer_train=cv_clean1_pooled_legacy.loc[~cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        validation=cv_clean1_pooled_legacy.loc[cv_clean1_pooled_legacy["cv_fold"].eq(fold_id)].copy()
        stratification=pd.cut(outer_train[TARGET_COL],bins=CV_SOH_EDGES,include_lowest=True,right=True).astype(str)
        core_indices,calibration_indices=train_test_split(outer_train.index,test_size=UNCERTAINTY_CALIBRATION_FRACTION,random_state=UNCERTAINTY_RANDOM_STATE+fold_id,stratify=stratification)
        core=outer_train.loc[core_indices].copy(); calibration=outer_train.loc[calibration_indices].copy()
        crossfit_pred=inner_cross_fitted_global_predictions(core,model_features,100+fold_id)
        core["delta_target"]=core[TARGET_COL]-crossfit_pred
        X_core=core[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan); medians=X_core.median(axis=0)
        X_core=X_core.fillna(medians); X_cal=calibration[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(medians)
        X_val=validation[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(medians)
        global_model=ExtraTreesRegressor(**MODEL_PARAMETERS); global_model.fit(X_core,core[TARGET_COL].to_numpy(dtype=float))
        gate_model=ExtraTreesClassifier(**GATE_PARAMETERS); gate_model.fit(X_core,core[TARGET_COL].le(EXPERT_SUPPORT_MAX).astype(int))
        expert_core=core.loc[core[TARGET_COL].le(EXPERT_SUPPORT_MAX)].copy(); X_expert=expert_core[model_features].apply(pd.to_numeric,errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(medians)
        expert_model=ExtraTreesRegressor(**MODEL_PARAMETERS); expert_model.fit(X_expert,expert_core["delta_target"].to_numpy(dtype=float))
        global_cal=global_model.predict(X_cal); global_val=global_model.predict(X_val)
        gate_cal=gate_model.predict_proba(X_cal)[:,list(gate_model.classes_).index(1)]; gate_val=gate_model.predict_proba(X_val)[:,list(gate_model.classes_).index(1)]
        delta_cal=expert_model.predict(X_cal); delta_val=expert_model.predict(X_val)
        ensemble_cal=global_cal+gate_cal*delta_cal; ensemble_val=global_val+gate_val*delta_val
        rng=np.random.default_rng(UNCERTAINTY_RANDOM_STATE+fold_id)
        global_indices=rng.integers(0,len(global_model.estimators_),size=UNCERTAINTY_MC_DRAWS); expert_indices=rng.integers(0,len(expert_model.estimators_),size=UNCERTAINTY_MC_DRAWS); gate_indices=rng.integers(0,len(gate_model.estimators_),size=UNCERTAINTY_MC_DRAWS)
        global_cal_draws=regression_tree_draws(global_model,X_cal,global_indices); global_val_draws=regression_tree_draws(global_model,X_val,global_indices)
        expert_cal_draws=regression_tree_draws(expert_model,X_cal,expert_indices); expert_val_draws=regression_tree_draws(expert_model,X_val,expert_indices)
        gate_cal_draws=gate_tree_draws(gate_model,X_cal,gate_indices); gate_val_draws=gate_tree_draws(gate_model,X_val,gate_indices)
        ensemble_cal_draws=global_cal_draws+gate_cal_draws*expert_cal_draws; ensemble_val_draws=global_val_draws+gate_val_draws*expert_val_draws
        y_cal=calibration[TARGET_COL].to_numpy(dtype=float); y_val=validation[TARGET_COL].to_numpy(dtype=float)
        for method,point_cal,point_val,draws_cal,draws_val in [("global",global_cal,global_val,global_cal_draws,global_val_draws),("delta_ensemble",ensemble_cal,ensemble_val,ensemble_cal_draws,ensemble_val_draws)]:
            u1_cal=np.std(draws_cal,axis=0,ddof=1); u1_val=np.std(draws_val,axis=0,ddof=1)
            u1_low=np.quantile(draws_val,UNCERTAINTY_ALPHA/2,axis=0); u1_high=np.quantile(draws_val,1-UNCERTAINTY_ALPHA/2,axis=0)
            u1_90_low=np.quantile(draws_val,UNCERTAINTY_ALPHA_90/2,axis=0); u1_90_high=np.quantile(draws_val,1-UNCERTAINTY_ALPHA_90/2,axis=0)
            absolute_scores=np.abs(y_cal-point_cal); q_u2=conformal_quantile(absolute_scores); q_u2_90=conformal_quantile(absolute_scores,UNCERTAINTY_ALPHA_90)
            scale_cal=np.maximum(u1_cal,UNCERTAINTY_SCALE_FLOOR); scale_val=np.maximum(u1_val,UNCERTAINTY_SCALE_FLOOR)
            normalized_scores=absolute_scores/scale_cal; q_u3=conformal_quantile(normalized_scores); q_u3_90=conformal_quantile(normalized_scores,UNCERTAINTY_ALPHA_90)
            calibration_regime=pd.cut(gate_cal,bins=MONDRIAN_GATE_EDGES,labels=MONDRIAN_GATE_LABELS,right=False).astype(str)
            validation_regime=pd.cut(gate_val,bins=MONDRIAN_GATE_EDGES,labels=MONDRIAN_GATE_LABELS,right=False).astype(str)
            mondrian_counts={}; q_u3m_95={}; q_u3m_90={}
            for regime in MONDRIAN_GATE_LABELS:
                mask=calibration_regime==regime; mondrian_counts[regime]=int(mask.sum())
                if mask.sum()>=MONDRIAN_MIN_CALIBRATION:
                    q_u3m_95[regime]=conformal_quantile(normalized_scores[mask],UNCERTAINTY_ALPHA)
                    q_u3m_90[regime]=conformal_quantile(normalized_scores[mask],UNCERTAINTY_ALPHA_90)
                else:
                    q_u3m_95[regime]=q_u3; q_u3m_90[regime]=q_u3_90
            event_q_u3m_95=np.asarray([q_u3m_95[regime] for regime in validation_regime]); event_q_u3m_90=np.asarray([q_u3m_90[regime] for regime in validation_regime])
            result=pd.DataFrame({"fold":fold_id,"method":method,"event_id":validation["event_id"].to_numpy(),"car":validation["car"].to_numpy(),
                "mileage":validation["mileage"].to_numpy(),"soh_real":y_val,"soh_pred":point_val,"gate_probability":gate_val,"mondrian_regime":validation_regime,
                "u1_std":u1_val,"u1_low":u1_low,"u1_high":u1_high,"u2_low":point_val-q_u2,"u2_high":point_val+q_u2,
                "u3_low":point_val-q_u3*scale_val,"u3_high":point_val+q_u3*scale_val,
                "u1_90_low":u1_90_low,"u1_90_high":u1_90_high,"u2_90_low":point_val-q_u2_90,"u2_90_high":point_val+q_u2_90,
                "u3_90_low":point_val-q_u3_90*scale_val,"u3_90_high":point_val+q_u3_90*scale_val,
                "u3m_low":point_val-event_q_u3m_95*scale_val,"u3m_high":point_val+event_q_u3m_95*scale_val,
                "u3m_90_low":point_val-event_q_u3m_90*scale_val,"u3m_90_high":point_val+event_q_u3m_90*scale_val}); prediction_frames.append(result)
            fold_audits.append({"fold":fold_id,"method":method,"n_core":len(core),"n_calibration":len(calibration),"n_validation":len(validation),"n_expert_core":len(expert_core),
                "q_u2_95":q_u2,"q_u3_95":q_u3,"q_u2_90":q_u2_90,"q_u3_90":q_u3_90,
                "mondrian_counts_json":json.dumps(mondrian_counts,sort_keys=True),"q_u3m_95_json":json.dumps(q_u3m_95,sort_keys=True),"q_u3m_90_json":json.dumps(q_u3m_90,sort_keys=True),
                "calibration_u1_median":float(np.median(u1_cal)),"validation_u1_median":float(np.median(u1_val))})
        print(f"fold {fold_id}: core={len(core)}, calibração={len(calibration)}, especialista={len(expert_core)}, tempo={time.perf_counter()-fold_start:.1f}s")
        del global_model,gate_model,expert_model,global_cal_draws,global_val_draws,expert_cal_draws,expert_val_draws,gate_cal_draws,gate_val_draws,ensemble_cal_draws,ensemble_val_draws
    uncertainty_oof_predictions=pd.concat(prediction_frames,ignore_index=True); uncertainty_fold_audit=pd.DataFrame(fold_audits)
    if uncertainty_oof_predictions.duplicated(["method","event_id"]).any(): raise AssertionError("Predições de incerteza duplicadas")
    for frame,path in [(uncertainty_oof_predictions,UNCERTAINTY_PREDICTIONS_PATH),(uncertainty_fold_audit,UNCERTAINTY_FOLD_PATH)]:
        temporary=path.with_suffix(path.suffix+".tmp"); frame.to_csv(temporary,index=False); temporary.replace(path)
    manifest={"stage":"global_and_delta_uncertainty","stage_version":UNCERTAINTY_STAGE_VERSION,"created_at_utc":datetime.now(timezone.utc).isoformat(),"completed":True,
        "parameters":uncertainty_parameters,"input_sha256":uncertainty_inputs,"predictions_shape":list(uncertainty_oof_predictions.shape),
        "predictions_sha256":sha256_file(UNCERTAINTY_PREDICTIONS_PATH),"fold_audit_sha256":sha256_file(UNCERTAINTY_FOLD_PATH),"elapsed_seconds":time.perf_counter()-total_start}
    temporary=UNCERTAINTY_MANIFEST_PATH.with_suffix(".json.tmp");temporary.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8");temporary.replace(UNCERTAINTY_MANIFEST_PATH);uncertainty_cache_status="construído"
print("Cache de incertezas:",uncertainty_cache_status)


In [ ]:
uncertainty_rows=[]
for method,group in uncertainty_oof_predictions.groupby("method",sort=True):
    for scope,subset in [("global",group),("soh_le_094",group[group["soh_real"].le(DEGRADED_SOH_MAX)])]:
        row={"method":method,"scope":scope,"n_events":len(subset),**regression_metrics(subset["soh_real"],subset["soh_pred"])}
        for uncertainty in ["u1","u2","u3","u3m"]:
            low=subset[f"{uncertainty}_low"];high=subset[f"{uncertainty}_high"];covered=subset["soh_real"].between(low,high,inclusive="both")
            row[f"{uncertainty}_coverage"]=covered.mean();row[f"{uncertainty}_mean_width"]=(high-low).mean();row[f"{uncertainty}_median_width"]=(high-low).median()
        uncertainty_rows.append(row)
uncertainty_summary=pd.DataFrame(uncertainty_rows)
efficiency_rows=[]
for method,group in uncertainty_oof_predictions.groupby("method",sort=True):
    for scope,subset in [("global",group),("soh_le_094",group[group["soh_real"].le(DEGRADED_SOH_MAX)])]:
        for nominal,suffix in [(0.90,"_90"),(0.95,"")]:
            for uncertainty in ["u1","u2","u3","u3m"]:
                low=subset[f"{uncertainty}{suffix}_low"];high=subset[f"{uncertainty}{suffix}_high"];covered=subset["soh_real"].between(low,high,inclusive="both")
                efficiency_rows.append({"method":method,"scope":scope,"nominal_coverage":nominal,"uncertainty":uncertainty,"n_events":len(subset),
                    "empirical_coverage":covered.mean(),"mean_width":(high-low).mean(),"median_width":(high-low).median()})
uncertainty_efficiency_comparison=pd.DataFrame(efficiency_rows)
threshold_rows=[]
for method,group in uncertainty_oof_predictions.groupby("method",sort=True):
    for uncertainty in ["u1","u2","u3","u3m"]:
        low=group[f"{uncertainty}_low"];high=group[f"{uncertainty}_high"]
        status=np.select([high.lt(DEGRADED_SOH_MAX),low.gt(DEGRADED_SOH_MAX)],["below_094","above_094"],default="crosses_094")
        counts=pd.Series(status).value_counts(); threshold_rows.append({"method":method,"uncertainty":uncertainty,
            "below_094":int(counts.get("below_094",0)),"crosses_094":int(counts.get("crosses_094",0)),"above_094":int(counts.get("above_094",0))})
threshold_interval_audit=pd.DataFrame(threshold_rows)
mondrian_rows=[]
for (method,regime),subset in uncertainty_oof_predictions.groupby(["method","mondrian_regime"],sort=True):
    for nominal,suffix in [(0.90,"_90"),(0.95,"")]:
        low=subset[f"u3m{suffix}_low"];high=subset[f"u3m{suffix}_high"];covered=subset["soh_real"].between(low,high,inclusive="both")
        mondrian_rows.append({"method":method,"regime":regime,"nominal_coverage":nominal,"n_events":len(subset),
            "empirical_coverage":covered.mean(),"mean_width":(high-low).mean(),"median_width":(high-low).median(),"n_true_soh_le_094":int(subset["soh_real"].le(DEGRADED_SOH_MAX).sum())})
mondrian_regime_audit=pd.DataFrame(mondrian_rows)
display(uncertainty_fold_audit);display(uncertainty_summary);display(uncertainty_efficiency_comparison);display(mondrian_regime_audit);display(threshold_interval_audit)


## 29. Cobertura e largura do U3 por faixa de SOH

As tabelas abaixo auditam o **U3 puro (conformal normalizado)** nos níveis nominais de **95% e 90%**, em faixas do SOH verdadeiro. Os bins são usados somente após a predição, para avaliação; portanto, não participam do treinamento nem da calibração. São apresentados separadamente os modelos `global` e `delta_ensemble`.


In [ ]:
def build_u3_table_by_soh_bin(predictions, bin_width):
    required = {"method", "soh_real", "u3_low", "u3_high", "u3_90_low", "u3_90_high"}
    missing = required.difference(predictions.columns)
    if missing:
        raise KeyError(f"Colunas ausentes para a auditoria U3: {sorted(missing)}")
    if bin_width <= 0:
        raise ValueError("bin_width deve ser positivo")

    data = predictions[list(required)].copy()
    interval_columns = ["soh_real", "u3_low", "u3_high", "u3_90_low", "u3_90_high"]
    for column in interval_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    if data[interval_columns].isna().any().any():
        raise ValueError("Há valores ausentes ou não numéricos nos resultados U3")

    lower_edge = np.floor(data["soh_real"].min() / bin_width) * bin_width
    upper_edge = np.ceil(data["soh_real"].max() / bin_width) * bin_width + bin_width
    edges = np.round(np.arange(lower_edge, upper_edge + bin_width / 2, bin_width), 10)
    data["soh_bin"] = pd.cut(data["soh_real"], bins=edges, right=False, include_lowest=True)
    data["u3_95_covered"] = data["soh_real"].between(data["u3_low"], data["u3_high"], inclusive="both")
    data["u3_95_width"] = data["u3_high"] - data["u3_low"]
    data["u3_90_covered"] = data["soh_real"].between(data["u3_90_low"], data["u3_90_high"], inclusive="both")
    data["u3_90_width"] = data["u3_90_high"] - data["u3_90_low"]

    rows = []
    for (method, interval), group in data.groupby(["method", "soh_bin"], observed=True, sort=True):
        rows.append({
            "method": method,
            "soh_bin": f"[{interval.left:.3f}, {interval.right:.3f})",
            "soh_lower": float(interval.left),
            "soh_upper": float(interval.right),
            "n_events": int(len(group)),
            "u3_95_coverage": float(group["u3_95_covered"].mean()),
            "u3_95_mean_width": float(group["u3_95_width"].mean()),
            "u3_95_median_width": float(group["u3_95_width"].median()),
            "u3_90_coverage": float(group["u3_90_covered"].mean()),
            "u3_90_mean_width": float(group["u3_90_width"].mean()),
            "u3_90_median_width": float(group["u3_90_width"].median()),
        })
    result = pd.DataFrame(rows)
    if int(result["n_events"].sum()) != len(data):
        raise AssertionError("Nem todos os resultados U3 foram atribuídos a um bin")
    return result

u3_table_bins_0005 = build_u3_table_by_soh_bin(uncertainty_oof_predictions, 0.005)
u3_table_bins_002 = build_u3_table_by_soh_bin(uncertainty_oof_predictions, 0.02)

u3_table_bins_0005.to_csv(OUTPUT_DIR / "u3_coverage_width_by_soh_bin_0005.csv", index=False)
u3_table_bins_002.to_csv(OUTPUT_DIR / "u3_coverage_width_by_soh_bin_002.csv", index=False)

print("Tabela 1 — U3 puro a 95% e 90% por bins de 0,005")
display(u3_table_bins_0005)
print("Tabela 2 — U3 puro a 95% e 90% por bins de 0,02")
display(u3_table_bins_002)


## Fim do pipeline compartilhado — com mileage

Esta versão reproduz o processamento desde `data/data_real_EV_charge.pkl`, os modelos e as incertezas U1, U2, U3 e U3-Mondrian. A execução termina deliberadamente após a auditoria das incertezas. Consulte `README.md` para ambiente, ordem de execução, checkpoints e resultados esperados.